In [3]:
import pandas as pd
import re

# ── Paste your CSV path here ──────────────────────────────────────────────────
MRI_CSV = '../data/mri_reports.csv'
# ─────────────────────────────────────────────────────────────────────────────

STRUCTURAL_TYPES = [
    'MRI BRAIN',
    'MRI BRAIN WITH AND WITHOUT CONTRAST',
    'MRI BRAIN WITHOUT CONTRAST',
]

SECTION_SPLIT_PATTERN = re.compile(
    r'\n(?=(?:BRAIN MRI|HEAD MRA|NECK MRA|MRI BRAIN AND NECK'
    r'|SPINE|CERVICAL|THORACIC|LUMBAR|STRUCTURAL MRI|FUNCTIONAL MRI)\s*[:\n])',
    re.IGNORECASE
)
BRAIN_SECTION_PATTERN = re.compile(r'^(?:BRAIN MRI|MRI BRAIN|STRUCTURAL MRI)', re.IGNORECASE)
INLINE_HEADER_PATTERN = re.compile(r'^(?:BRAIN MRI|HEAD MRA|MRI BRAIN)\s*:?\s*\n', re.IGNORECASE)

BOILERPLATE_PATTERN = re.compile(
    r'forwarded to an automated communication'
    r'|findings were (?:discussed|communicated)'
    r'|carotid stenosis reference'
    r'|i,? the (?:attending|teaching) physician'
    r'|alert notification of critical'
    r'|critical results were communicated'
    r'|this report has been'
    r'|electronically notify'
    r'|study (?:terminated|aborted|discontinued) (?:early|due to|at patient)',
    re.IGNORECASE
)
HEADER_PATTERN = re.compile(
    r'^(?:brain mri|head mra|neck mra|mri brain|intracranial mra'
    r'|anterior circulation|posterior circulation|collateral circulation'
    r'|aortic\s+\[redacted\].*origin of major cervical'
    r'|impression|technique|comparison|indication|methodology'
    r'|detail|findings|neck):?\s*$',
    re.IGNORECASE
)
NEGATIVE_PATTERN = re.compile(
    r'^(?:there is |there are )?(?:no evidence of|no acute|no new|no abnormal'
    r'|no significant|unremarkable|within normal limits|no hemorrhage'
    r'|no infarct|no mass effect|no hydrocephalus|no midline shift'
    r'|patent|intact|clear|stable and unremarkable)'
    r'|^the (?:major |main )?(?:intracranial )?(?:flow voids|arterial flow voids|vessels).{0,60}(?:intact|preserved|patent)'
    r'|^(?:extracranial structures|paranasal sinuses|mastoid|orbits).{0,80}(?:clear|normal|unremarkable)',
    re.IGNORECASE
)
EXTRACRANIAL_PATTERN = re.compile(
    r'^(?:paranasal sinuses|mastoid air cells|orbits|skull base'
    r'|extracranial soft tissues|cervical|vertebral arter'
    r'|aortic|subclavian|carotid)',
    re.IGNORECASE
)

def is_noise(text):
    if len(text.strip()) < 30: return True
    if HEADER_PATTERN.match(text): return True
    if BOILERPLATE_PATTERN.search(text): return True
    return False

def is_negative(text): return bool(NEGATIVE_PATTERN.match(text))
def is_extracranial(text): return bool(EXTRACRANIAL_PATTERN.match(text))

def _extract_brain_section(text):
    if not SECTION_SPLIT_PATTERN.search(text):
        return text
    sections = SECTION_SPLIT_PATTERN.split(text)
    brain_sections = [s for s in sections if BRAIN_SECTION_PATTERN.match(s.strip())]
    return brain_sections[0] if brain_sections else sections[0]

def split_report(text):
    if pd.isna(text): return []
    text = _extract_brain_section(text)
    paragraphs = re.split(r'\n[ \t]*\n', text)
    result = []
    for p in paragraphs:
        p = p.strip()
        p = INLINE_HEADER_PATTERN.sub('', p).strip()
        if len(p) >= 30:
            result.append(p)
    return result

def split_report_old(text):
    if pd.isna(text): return []
    return [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]

# ── Load ──────────────────────────────────────────────────────────────────────
print('Loading data...')
mri = pd.read_csv(MRI_CSV, engine='python', on_bad_lines='skip')
valid = mri[mri['findings'].notna() & ~mri['findings'].str.strip().str.lower().isin(['not available','none',''])]
structural = valid[valid['Type'].isin(STRUCTURAL_TYPES)].copy()
print(f'Structural MRI rows: {len(structural):,}\n')

# ── TEST 1: Overall paragraph count comparison ────────────────────────────────
print('=' * 60)
print('TEST 1: Paragraph count — old vs new splitter')
print('=' * 60)
structural['old_count'] = structural['findings'].apply(lambda t: len(split_report_old(t)))
structural['new_count'] = structural['findings'].apply(lambda t: len(split_report(t)))

for label, col in [('Old splitter', 'old_count'), ('New splitter', 'new_count')]:
    s = structural[col]
    print(f'\n{label}:')
    print(f'  Mean:   {s.mean():.2f}')
    print(f'  Median: {s.median():.0f}')
    print(f'  Max:    {s.max()}')
    print(f'  >20 paragraphs (over-split): {(s > 20).sum():,} reports ({100*(s>20).mean():.1f}%)')

improved = (structural['new_count'] < structural['old_count']).sum()
regressed = (structural['new_count'] > structural['old_count']).sum()
print(f'\nImproved (fewer paragraphs):  {improved:,} reports ({100*improved/len(structural):.1f}%)')
print(f'Regressed (more paragraphs):  {regressed:,} reports ({100*regressed/len(structural):.1f}%)')

# ── TEST 2: Multi-section report handling ─────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 2: Multi-section report handling (BRAIN MRI + HEAD MRA etc.)')
print('=' * 60)
multi_section = structural[SECTION_SPLIT_PATTERN.search(structural['findings'].fillna('').astype(str)).values
    if False else structural['findings'].str.contains(
        r'HEAD MRA|NECK MRA|MRI BRAIN AND NECK', case=False, regex=True, na=False
    )]
print(f'Multi-section reports detected: {len(multi_section):,}')
if len(multi_section) > 0:
    print(f'  Old avg paragraphs: {multi_section["old_count"].mean():.1f}')
    print(f'  New avg paragraphs: {multi_section["new_count"].mean():.1f}')
    print('\nSample multi-section report — new split output:')
    sample = multi_section.iloc[0]
    paras = split_report(sample['findings'])
    for i, p in enumerate(paras[:5]):
        print(f'  [{i+1}] {p[:120]}')
    if len(paras) > 5:
        print(f'  ... ({len(paras)-5} more paragraphs)')

# ── TEST 3: Noise filter performance ─────────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 3: Noise filter performance')
print('=' * 60)
structural['paragraph_list'] = structural['findings'].apply(split_report)
exploded = (
    structural.explode('paragraph_list')
    .rename(columns={'paragraph_list': 'paragraph'})
    .dropna(subset=['paragraph'])
    .reset_index(drop=True)
)
noise_mask = exploded['paragraph'].apply(is_noise)
filtered = exploded[~noise_mask].copy().reset_index(drop=True)
filtered['is_negative'] = filtered['paragraph'].apply(is_negative)
filtered['is_extracranial'] = filtered['paragraph'].apply(is_extracranial)
positive = (~filtered['is_negative'] & ~filtered['is_extracranial']).sum()

print(f'Total paragraphs after split:    {len(exploded):,}')
print(f'Dropped (noise):                 {noise_mask.sum():,} ({100*noise_mask.mean():.1f}%)')
print(f'Remaining:                       {len(filtered):,}')
print(f'  - Negative/normal (tagged):    {filtered["is_negative"].sum():,}')
print(f'  - Extracranial (tagged):       {filtered["is_extracranial"].sum():,}')
print(f'  - Positive brain findings:     {positive:,}')

# ── TEST 4: Sample output across complexity tiers ─────────────────────────────
print('\n' + '=' * 60)
print('TEST 4: Sample output — short / medium / long reports')
print('=' * 60)
structural['new_count'] = structural['findings'].apply(lambda t: len(split_report(t)))
for label, mask in [
    ('SHORT (1-2 paragraphs)',    structural['new_count'].between(1, 2)),
    ('MEDIUM (6-8 paragraphs)',   structural['new_count'].between(6, 8)),
    ('LONG (15+ paragraphs)',     structural['new_count'] >= 15),
]:
    subset = structural[mask]
    if len(subset) == 0:
        continue
    row = subset.sample(1, random_state=42).iloc[0]
    paras = split_report(row['findings'])
    print(f'\n--- {label} | {len(paras)} paragraphs ---')
    for i, p in enumerate(paras[:4]):
        neg = '[NEG] ' if is_negative(p) else '[POS] '
        ext = '[EXT] ' if is_extracranial(p) else ''
        print(f'  {neg}{ext}{p[:130]}')
    if len(paras) > 4:
        print(f'  ... ({len(paras)-4} more)')

# ── TEST 5: Spot check — 20 random positive findings ─────────────────────────
print('\n' + '=' * 60)
print('TEST 5: 20 random positive brain findings (manual review)')
print('=' * 60)
pos_sample = filtered[~filtered['is_negative'] & ~filtered['is_extracranial']].sample(20, random_state=99)
for i, (_, row) in enumerate(pos_sample.iterrows()):
    print(f'\n[{i+1}] {row["paragraph"][:200]}')

print('\n' + '=' * 60)
print('ALL TESTS COMPLETE')
print('=' * 60)

Loading data...
Structural MRI rows: 36,924

TEST 1: Paragraph count — old vs new splitter

Old splitter:
  Mean:   9.05
  Median: 8
  Max:    88
  >20 paragraphs (over-split): 1,475 reports (4.0%)

New splitter:
  Mean:   7.94
  Median: 7
  Max:    70
  >20 paragraphs (over-split): 483 reports (1.3%)

Improved (fewer paragraphs):  11,216 reports (30.4%)
Regressed (more paragraphs):  0 reports (0.0%)

TEST 2: Multi-section report handling (BRAIN MRI + HEAD MRA etc.)
Multi-section reports detected: 1,725
  Old avg paragraphs: 19.6
  New avg paragraphs: 7.0

Sample multi-section report — new split output:
  [1] Brain Parenchyma: There are extensive regions of T2 prolongation in the white matter likely manifestation of chronic sma
  [2] Ventricular System and Extra-Axial Spaces: The ventricles and cortical sulci are prominent. Cavum septum pellucidum is i
  [3] Skull Base, [REDACTED], and Visualized Upper Cervical Spine: Normal.
  [4] Paranasal Sinuses, Mastoids, and Orbits: Mucosal thick

In [4]:
import pandas as pd
import re

# ── Paste your CSV path here ──────────────────────────────────────────────────
MRI_CSV = '../data/mri_reports.csv'
# ─────────────────────────────────────────────────────────────────────────────

STRUCTURAL_TYPES = [
    'MRI BRAIN',
    'MRI BRAIN WITH AND WITHOUT CONTRAST',
    'MRI BRAIN WITHOUT CONTRAST',
]

# ── Filters ───────────────────────────────────────────────────────────────────

BOILERPLATE_PATTERN = re.compile(
    r'forwarded to an automated communication'
    r'|findings were (?:discussed|communicated)'
    r'|alert notification of critical'
    r'|critical results were communicated'
    r'|this report has been'
    r'|electronically notify'
    r'|per epic note'
    r'|carotid stenosis reference'
    r'|(?:\[redacted\][\w\s,\.]*)?(?:md|do|phd|np|pa)[\s,]+the (?:attending|teaching) physician'
    r'|i,? the (?:attending|teaching) physician'
    r'|have reviewed the images and agree'
    r'|study (?:terminated|aborted|discontinued) (?:early|due to|at patient)'
    r'|post contrast[:\s]+following (?:iv|intravenous) administration'
    r'|following (?:iv|intravenous) administration of (?:gadolinium|contrast)',
    re.IGNORECASE
)

HEADER_PATTERN = re.compile(
    r'^(?:brain mri|head mra|neck mra|mri brain|intracranial mra'
    r'|anterior circulation|posterior circulation|collateral circulation'
    r'|aortic\s+\[redacted\].*origin of major cervical'
    r'|impression|technique|comparison|indication|methodology'
    r'|detail|findings|neck'
    r'|ventricular system[\w\s\-]+:'
    r'|extra-axial spaces?:'
    r'|brain parenchyma:'
    r'|vascular structures?:'
    r'|flow.related signal):?\s*$',
    re.IGNORECASE
)

NEGATIVE_PATTERN = re.compile(
    r'^(?:there (?:is|are) )?(?:no evidence of|no acute|no new|no abnormal'
    r'|no significant|unremarkable|within normal limits|no hemorrhage'
    r'|no infarct|no mass effect|no hydrocephalus|no midline shift'
    r'|no space.occupying|no enhancing lesion|no abnormal enhancement'
    r'|no restricted diffusion|no acute intracranial)'
    r'|^the (?:major |main )?(?:intracranial )?(?:flow voids|arterial flow voids|vessels).{0,60}(?:intact|preserved|patent)'
    r'|^the (?:visualized )?(?:paranasal sinuses|mastoid air cells?|orbits|bones and extracranial|extracranial soft tissues).{0,80}(?:clear|normal|unremarkable|unchanged)'
    r'|^the (?:remaining )?ventricles? (?:are|is) (?:stable|normal|unremarkable).{0,60}(?:no hydrocephalus|no (?:evidence of )?mass|unchanged)'
    r'|^(?:extracranial structures|paranasal sinuses|mastoid).{0,80}(?:clear|normal|unremarkable)'
    r'|^(?:post contrast|following contrast).{0,60}no (?:abnormal|significant)',
    re.IGNORECASE
)

EXTRACRANIAL_PATTERN = re.compile(
    r'^(?:paranasal sinuses|mastoid air cells?|orbits|skull base'
    r'|extracranial soft tissues|cervical|vertebral arter'
    r'|aortic|subclavian|carotid)'
    r'|^the (?:bones and )?extracranial'
    r'|^the (?:visualized )?(?:paranasal sinuses|mastoid|orbits)'
    r'|^(?:partial|bilateral|mild|moderate|severe)?\s*(?:opacification|thickening|mucosal).{0,40}(?:sinus|mastoid|orbit)'
    r'|^the (?:superior sagittal|transverse|sigmoid|straight|cavernous) sinus',
    re.IGNORECASE
)

def is_noise(text):
    if len(text.strip()) < 30: return True
    if HEADER_PATTERN.match(text): return True
    if BOILERPLATE_PATTERN.search(text): return True
    return False

def is_negative(text): return bool(NEGATIVE_PATTERN.match(text))
def is_extracranial(text): return bool(EXTRACRANIAL_PATTERN.match(text))

# ── Splitter ──────────────────────────────────────────────────────────────────

SECTION_SPLIT_PATTERN = re.compile(
    r'\n(?=(?:BRAIN MRI|HEAD MRA|NECK MRA|MRI BRAIN AND NECK'
    r'|SPINE|CERVICAL|THORACIC|LUMBAR|STRUCTURAL MRI|FUNCTIONAL MRI)\s*[:\n])',
    re.IGNORECASE
)
BRAIN_SECTION_PATTERN = re.compile(r'^(?:BRAIN MRI|MRI BRAIN|STRUCTURAL MRI)', re.IGNORECASE)
INLINE_HEADER_PATTERN = re.compile(r'^(?:BRAIN MRI|HEAD MRA|MRI BRAIN)\s*:?\s*\n', re.IGNORECASE)

def _extract_brain_section(text):
    if not SECTION_SPLIT_PATTERN.search(text):
        return text
    sections = SECTION_SPLIT_PATTERN.split(text)
    brain_sections = [s for s in sections if BRAIN_SECTION_PATTERN.match(s.strip())]
    return brain_sections[0] if brain_sections else sections[0]

def split_report(text):
    if pd.isna(text): return []
    text = _extract_brain_section(text)
    paragraphs = re.split(r'\n[ \t]*\n', text)
    result = []
    for p in paragraphs:
        p = p.strip()
        p = INLINE_HEADER_PATTERN.sub('', p).strip()
        if len(p) >= 30:
            result.append(p)
    return result

def split_report_old(text):
    if pd.isna(text): return []
    return [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]

# ── Load ──────────────────────────────────────────────────────────────────────
print('Loading data...')
mri = pd.read_csv(MRI_CSV, engine='python', on_bad_lines='skip')
valid = mri[mri['findings'].notna() & ~mri['findings'].str.strip().str.lower().isin(['not available','none',''])]
structural = valid[valid['Type'].isin(STRUCTURAL_TYPES)].copy()
print(f'Structural MRI rows: {len(structural):,}\n')

# ── TEST 1: Paragraph count comparison ───────────────────────────────────────
print('=' * 60)
print('TEST 1: Paragraph count — old vs new splitter')
print('=' * 60)
structural['old_count'] = structural['findings'].apply(lambda t: len(split_report_old(t)))
structural['new_count'] = structural['findings'].apply(lambda t: len(split_report(t)))

for label, col in [('Old splitter', 'old_count'), ('New splitter', 'new_count')]:
    s = structural[col]
    print(f'\n{label}:')
    print(f'  Mean:   {s.mean():.2f}')
    print(f'  Median: {s.median():.0f}')
    print(f'  Max:    {s.max()}')
    print(f'  >20 paragraphs (over-split): {(s > 20).sum():,} ({100*(s>20).mean():.1f}%)')

improved = (structural['new_count'] < structural['old_count']).sum()
regressed = (structural['new_count'] > structural['old_count']).sum()
print(f'\nImproved: {improved:,} ({100*improved/len(structural):.1f}%) | Regressed: {regressed:,} ({100*regressed/len(structural):.1f}%)')

# ── TEST 2: Multi-section report handling ─────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 2: Multi-section report handling')
print('=' * 60)
multi_section = structural[structural['findings'].str.contains(
    r'HEAD MRA|NECK MRA|MRI BRAIN AND NECK', case=False, regex=True, na=False
)]
print(f'Multi-section reports: {len(multi_section):,}')
if len(multi_section) > 0:
    print(f'  Old avg paragraphs: {multi_section["old_count"].mean():.1f}')
    print(f'  New avg paragraphs: {multi_section["new_count"].mean():.1f}')

# ── TEST 3: Filter performance ────────────────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 3: Filter performance')
print('=' * 60)
structural['paragraph_list'] = structural['findings'].apply(split_report)
exploded = (
    structural.explode('paragraph_list')
    .rename(columns={'paragraph_list': 'paragraph'})
    .dropna(subset=['paragraph'])
    .reset_index(drop=True)
)
noise_mask = exploded['paragraph'].apply(is_noise)
filtered = exploded[~noise_mask].copy().reset_index(drop=True)
filtered['is_negative'] = filtered['paragraph'].apply(is_negative)
filtered['is_extracranial'] = filtered['paragraph'].apply(is_extracranial)
positive = (~filtered['is_negative'] & ~filtered['is_extracranial']).sum()

print(f'Total paragraphs:              {len(exploded):,}')
print(f'Dropped (noise):               {noise_mask.sum():,} ({100*noise_mask.mean():.1f}%)')
print(f'Remaining:                     {len(filtered):,}')
print(f'  - Negative/normal:           {filtered["is_negative"].sum():,}')
print(f'  - Extracranial:              {filtered["is_extracranial"].sum():,}')
print(f'  - Positive brain findings:   {positive:,}')

# ── TEST 4: Regression check on known bad cases from v1 ──────────────────────
print('\n' + '=' * 60)
print('TEST 4: Regression check on previously failing cases')
print('=' * 60)
known_cases = [
    ("Extracranial starting with 'The bones'",
     "The bones and extracranial soft tissues are unchanged.",
     'extracranial'),
    ("Physician sign-off with [REDACTED]",
     "[REDACTED] MD, the teaching physician, have reviewed the images and agree with the report as written.",
     'noise'),
    ("Partial opacification of sinuses",
     "Partial opacification of the maxillary sinuses bilaterally. The mastoid air cells and orbits are unremarkable.",
     'extracranial'),
    ("Ventricles stable no hydrocephalus",
     "The remaining ventricles are stable in size and configuration. There is no hydrocephalus.",
     'negative'),
    ("Post contrast technique note",
     "Post contrast: Following IV administration of gadolinium chelate, no abnormal regions of enhancement are demonstrated.",
     'noise'),
    ("Per Epic note",
     "Per Epic note, the patient subsequently underwent evacuation of the bifrontal fluid collection where purulent material was noted.",
     'noise'),
    ("Dural sinuses",
     "The superior sagittal sinus, internal cerebral veins, inferior sagittal sinus, straight sinus, transverse sinus, sigmoid sinus are patent.",
     'extracranial'),
]

all_passed = True
for desc, text, expected in known_cases:
    if expected == 'noise':
        result = is_noise(text)
        label = 'noise'
    elif expected == 'negative':
        result = is_negative(text)
        label = 'negative'
    elif expected == 'extracranial':
        result = is_extracranial(text)
        label = 'extracranial'

    status = 'PASS' if result else 'FAIL'
    if not result:
        all_passed = False
    print(f'  [{status}] {desc}')

print(f'\n{"All regression tests passed!" if all_passed else "Some tests failed -- check patterns above."}')

# ── TEST 5: 20 random positive findings ──────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 5: 20 random positive brain findings (manual review)')
print('=' * 60)
pos = filtered[~filtered['is_negative'] & ~filtered['is_extracranial']]
for i, (_, row) in enumerate(pos.sample(20, random_state=99).iterrows()):
    print(f'\n[{i+1}] {row["paragraph"][:200]}')

print('\n' + '=' * 60)
print('ALL TESTS COMPLETE')
print('=' * 60)

Loading data...
Structural MRI rows: 36,924

TEST 1: Paragraph count — old vs new splitter

Old splitter:
  Mean:   9.05
  Median: 8
  Max:    88
  >20 paragraphs (over-split): 1,475 (4.0%)

New splitter:
  Mean:   7.94
  Median: 7
  Max:    70
  >20 paragraphs (over-split): 483 (1.3%)

Improved: 11,216 (30.4%) | Regressed: 0 (0.0%)

TEST 2: Multi-section report handling
Multi-section reports: 1,725
  Old avg paragraphs: 19.6
  New avg paragraphs: 7.0

TEST 3: Filter performance
Total paragraphs:              293,309
Dropped (noise):               7,964 (2.7%)
Remaining:                     285,345
  - Negative/normal:           56,668
  - Extracranial:              25,130
  - Positive brain findings:   215,940

TEST 4: Regression check on previously failing cases
  [PASS] Extracranial starting with 'The bones'
  [PASS] Physician sign-off with [REDACTED]
  [PASS] Partial opacification of sinuses
  [PASS] Ventricles stable no hydrocephalus
  [PASS] Post contrast technique note
  [PASS] 

In [5]:
import pandas as pd
import re
import random

# ── Config ────────────────────────────────────────────────────────────────────
MRI_CSV = '../data/mri_reports.csv'
# ─────────────────────────────────────────────────────────────────────────────

STRUCTURAL_TYPES = [
    'MRI BRAIN',
    'MRI BRAIN WITH AND WITHOUT CONTRAST',
    'MRI BRAIN WITHOUT CONTRAST',
]

# ── Filters (v3) ──────────────────────────────────────────────────────────────

BOILERPLATE_PATTERN = re.compile(
    r'forwarded to an automated communication'
    r'|findings were (?:discussed|communicated)'
    r'|alert notification of critical'
    r'|critical results were communicated'
    r'|this report has been'
    r'|electronically notify'
    r'|per epic note'
    r'|carotid stenosis reference'
    r'|(?:\[redacted\][\w\s,\.]*)?(?:md|do|phd|np|pa)[\s,]+the (?:attending|teaching) physician'
    r'|i,? the (?:attending|teaching) physician'
    r'|have reviewed the images and agree'
    r'|study (?:terminated|aborted|discontinued) (?:early|due to|at patient)'
    r'|post contrast[:\s]+following (?:iv|intravenous) administration'
    r'|following (?:iv|intravenous) administration of (?:gadolinium|contrast)',
    re.IGNORECASE
)

HEADER_PATTERN = re.compile(
    r'^(?:brain mri|head mra|neck mra|mri brain|intracranial mra'
    r'|anterior circulation|posterior circulation|collateral circulation'
    r'|aortic\s+\[redacted\].*origin of major cervical'
    r'|impression|technique|comparison|indication|methodology'
    r'|detail|findings|neck'
    r'|ventricular system[\w\s\-]+:'
    r'|extra-axial spaces?:'
    r'|brain parenchyma:'
    r'|vascular system:'
    r'|vascular structures?:'
    r'|flow.related signal):?\s*$',
    re.IGNORECASE
)

NEGATIVE_PATTERN = re.compile(
    r'^(?:there (?:is|are) )?(?:no evidence of|no acute|no new|no abnormal'
    r'|no significant|unremarkable|within normal limits|no hemorrhage'
    r'|no infarct|no mass effect|no hydrocephalus|no midline shift'
    r'|no space.occupying|no enhancing lesion|no abnormal enhancement'
    r'|no restricted diffusion|no acute intracranial)'
    r'|^the (?:major |main )?(?:intracranial )?(?:flow voids|arterial flow voids|vessels).{0,60}(?:intact|preserved|patent)'
    r'|^the (?:visualized )?(?:paranasal sinuses|mastoid air cells?|orbits|bones and extracranial|extracranial soft tissues).{0,80}(?:clear|normal|unremarkable|unchanged)'
    r'|^the (?:remaining )?ventricles? (?:are|is) (?:stable|normal|unremarkable|unchanged).{0,80}(?:no hydrocephalus|no (?:evidence of )?mass|unchanged)'
    r'|^(?:the )?ventricles?[,\s]+sulci[,\s]+and (?:cisterns?|basal cisterns?).{0,60}(?:unremarkable|normal|stable|within normal limits)'
    r'|^ventricles?[,\s]+sulci[,\s]+and cisterns? are stable'
    r'|^(?:\[redacted\]\s+)?venous sinuses? (?:enhance|are|appear).{0,40}(?:normal|patent|intact|homogenous)'
    r'|^(?:extracranial structures|paranasal sinuses|mastoid).{0,80}(?:clear|normal|unremarkable)'
    r'|^(?:post contrast|following contrast).{0,60}no (?:abnormal|significant)'
    r'|^vascular system:.{0,80}(?:intact|preserved|patent|normal)',
    re.IGNORECASE
)

EXTRACRANIAL_PATTERN = re.compile(
    r'^(?:paranasal sinuses|mastoid air cells?|orbits|skull base'
    r'|extracranial soft tissues|cervical|vertebral arter'
    r'|aortic|subclavian|carotid|nasopharynx)'
    r'|^the (?:bones and )?extracranial'
    r'|^the (?:visualized )?(?:paranasal sinuses|mastoid|orbits)'
    r'|^(?:partial|bilateral|mild|moderate|severe)?\s*(?:opacification|thickening|mucosal).{0,40}(?:sinus|mastoid|orbit)'
    r'|(?:mastoid air cells?|middle ear cavity|maxillary sinus|sphenoid sinus|ethmoid).{0,30}(?:opacif|thicken|fluid|effusion)'
    r'|^the (?:superior sagittal|transverse|sigmoid|straight|cavernous) sinus'
    r'|^(?:mild|moderate|severe)?\s*(?:edema|mucosal thickening).{0,40}(?:nasopharynx|pharynx|soft tissue)',
    re.IGNORECASE
)

def is_noise(t):
    if len(t.strip()) < 30: return True
    if HEADER_PATTERN.match(t): return True
    if BOILERPLATE_PATTERN.search(t): return True
    return False

def is_negative(t): return bool(NEGATIVE_PATTERN.match(t))
def is_extracranial(t): return bool(EXTRACRANIAL_PATTERN.match(t))

# ── Splitter ──────────────────────────────────────────────────────────────────

SECTION_SPLIT_PATTERN = re.compile(
    r'\n(?=(?:BRAIN MRI|HEAD MRA|NECK MRA|MRI BRAIN AND NECK'
    r'|SPINE|CERVICAL|THORACIC|LUMBAR|STRUCTURAL MRI|FUNCTIONAL MRI)\s*[:\n])',
    re.IGNORECASE
)
BRAIN_SECTION_PATTERN = re.compile(r'^(?:BRAIN MRI|MRI BRAIN|STRUCTURAL MRI)', re.IGNORECASE)
INLINE_HEADER_PATTERN = re.compile(r'^(?:BRAIN MRI|HEAD MRA|MRI BRAIN)\s*:?\s*\n', re.IGNORECASE)

def _extract_brain_section(text):
    if not SECTION_SPLIT_PATTERN.search(text):
        return text
    sections = SECTION_SPLIT_PATTERN.split(text)
    brain = [s for s in sections if BRAIN_SECTION_PATTERN.match(s.strip())]
    return brain[0] if brain else sections[0]

def split_report(text):
    if pd.isna(text): return []
    text = _extract_brain_section(text)
    result = []
    for p in re.split(r'\n[ \t]*\n', text):
        p = INLINE_HEADER_PATTERN.sub('', p.strip()).strip()
        if len(p) >= 30:
            result.append(p)
    return result

def split_report_old(text):
    if pd.isna(text): return []
    return [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]

# ── Load ──────────────────────────────────────────────────────────────────────
print('Loading data...')
mri = pd.read_csv(MRI_CSV, engine='python', on_bad_lines='skip')
valid = mri[mri['findings'].notna() & ~mri['findings'].str.strip().str.lower().isin(['not available','none',''])]
structural = valid[valid['Type'].isin(STRUCTURAL_TYPES)].copy()
print(f'Structural MRI rows: {len(structural):,}\n')

# ── TEST 1 ────────────────────────────────────────────────────────────────────
print('=' * 60)
print('TEST 1: Paragraph count — old vs new splitter')
print('=' * 60)
structural['old_count'] = structural['findings'].apply(lambda t: len(split_report_old(t)))
structural['new_count'] = structural['findings'].apply(lambda t: len(split_report(t)))
for label, col in [('Old', 'old_count'), ('New', 'new_count')]:
    s = structural[col]
    print(f'\n{label}: mean={s.mean():.2f} | median={s.median():.0f} | max={s.max()} | >20para: {(s>20).sum():,} ({100*(s>20).mean():.1f}%)')
improved = (structural['new_count'] < structural['old_count']).sum()
regressed = (structural['new_count'] > structural['old_count']).sum()
print(f'\nImproved: {improved:,} ({100*improved/len(structural):.1f}%) | Regressed: {regressed:,} ({100*regressed/len(structural):.1f}%)')

# ── TEST 2 ────────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 2: Multi-section report handling')
print('=' * 60)
multi = structural[structural['findings'].str.contains(r'HEAD MRA|NECK MRA|MRI BRAIN AND NECK', case=False, regex=True, na=False)]
print(f'Multi-section reports: {len(multi):,} | Old avg: {multi["old_count"].mean():.1f} | New avg: {multi["new_count"].mean():.1f}')

# ── TEST 3 ────────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 3: Filter performance')
print('=' * 60)
structural['paragraph_list'] = structural['findings'].apply(split_report)
exploded = (
    structural.explode('paragraph_list')
    .rename(columns={'paragraph_list': 'paragraph'})
    .dropna(subset=['paragraph'])
    .reset_index(drop=True)
)
noise_mask = exploded['paragraph'].apply(is_noise)
filtered = exploded[~noise_mask].copy().reset_index(drop=True)
filtered['is_negative'] = filtered['paragraph'].apply(is_negative)
filtered['is_extracranial'] = filtered['paragraph'].apply(is_extracranial)
positive = (~filtered['is_negative'] & ~filtered['is_extracranial']).sum()
print(f'Total paragraphs:            {len(exploded):,}')
print(f'Dropped (noise):             {noise_mask.sum():,} ({100*noise_mask.mean():.1f}%)')
print(f'Remaining:                   {len(filtered):,}')
print(f'  - Negative/normal:         {filtered["is_negative"].sum():,}')
print(f'  - Extracranial:            {filtered["is_extracranial"].sum():,}')
print(f'  - Positive brain findings: {positive:,}')

# ── TEST 4: Regression check (fixed seed -- always same cases) ────────────────
print('\n' + '=' * 60)
print('TEST 4: Regression check on known failing cases (fixed)')
print('=' * 60)
cases = [
    ("Extracranial: 'The bones and extracranial...'",       "The bones and extracranial soft tissues are unchanged.", 'extracranial'),
    ("Noise: physician sign-off with [REDACTED]",           "[REDACTED] MD, the teaching physician, have reviewed the images and agree with the report as written.", 'noise'),
    ("Extracranial: partial opacification of sinuses",      "Partial opacification of the maxillary sinuses bilaterally. The mastoid air cells and orbits are unremarkable.", 'extracranial'),
    ("Negative: ventricles stable no hydrocephalus",        "The remaining ventricles are stable in size and configuration. There is no hydrocephalus.", 'negative'),
    ("Noise: post contrast technique note",                 "Post contrast: Following IV administration of gadolinium chelate, no abnormal regions of enhancement are demonstrated.", 'noise'),
    ("Noise: per Epic note",                                "Per Epic note, the patient subsequently underwent evacuation of the bifrontal fluid collection.", 'noise'),
    ("Extracranial: dural sinuses",                         "The superior sagittal sinus, internal cerebral veins, inferior sagittal sinus, straight sinus are patent.", 'extracranial'),
    ("Negative: vascular system normal",                    "Vascular System: Major intracranial flow voids present. Venous sinuses homogenous post contrast.", 'negative'),
    ("Extracranial: mastoid air cells mid-sentence",        "Patient status post left temporal craniotomy. Again opacification of left middle ear cavity and mastoid air cells.", 'extracranial'),
    ("Negative: ventricles sulci cisterns unremarkable",    "The ventricles, sulci, and basal cisterns are otherwise within normal limits.", 'negative'),
    ("Negative: ventricles sulci cisterns stable",          "Ventricles, sulci, and cisterns are stable in size and configuration.", 'negative'),
    ("Extracranial: venous sinuses enhance normally",       "[REDACTED] venous sinuses enhance normally.", 'negative'),
    ("Extracranial: nasopharynx mucosal thickening",        "There is mild edema and mucosal thickening in the nasopharynx anteriorly.", 'extracranial'),
    ("Extracranial: maxillary sinus asymmetry",             "Again seen is an asymmetrically diminutive appearance of the right maxillary sinus with lateral bowing.", 'extracranial'),
]

all_passed = True
for desc, text, expected in cases:
    if expected == 'noise':       result = is_noise(text)
    elif expected == 'negative':  result = is_negative(text)
    elif expected == 'extracranial': result = is_extracranial(text)
    status = 'PASS' if result else 'FAIL'
    if not result: all_passed = False
    print(f'  [{status}] {desc}')
print(f'\n{"All regression tests passed!" if all_passed else "Some tests FAILED -- check patterns."}')

# ── TEST 5: Random positive findings (new seed each run) ─────────────────────
print('\n' + '=' * 60)
print('TEST 5: 40 random positive brain findings (randomized each run)')
print('=' * 60)
pos = filtered[~filtered['is_negative'] & ~filtered['is_extracranial']]
sample = pos.sample(40)  # no random_state -- different every run
for i, (_, row) in enumerate(sample.iterrows()):
    print(f'\n[{i+1}] {row["paragraph"][:200]}')

print('\n' + '=' * 60)
print('ALL TESTS COMPLETE')
print('=' * 60)

Loading data...
Structural MRI rows: 36,924

TEST 1: Paragraph count — old vs new splitter

Old: mean=9.05 | median=8 | max=88 | >20para: 1,475 (4.0%)

New: mean=7.94 | median=7 | max=70 | >20para: 483 (1.3%)

Improved: 11,216 (30.4%) | Regressed: 0 (0.0%)

TEST 2: Multi-section report handling
Multi-section reports: 1,725 | Old avg: 19.6 | New avg: 7.0

TEST 3: Filter performance
Total paragraphs:            293,309
Dropped (noise):             7,964 (2.7%)
Remaining:                   285,345
  - Negative/normal:         61,987
  - Extracranial:            25,132
  - Positive brain findings: 210,619

TEST 4: Regression check on known failing cases (fixed)
  [PASS] Extracranial: 'The bones and extracranial...'
  [PASS] Noise: physician sign-off with [REDACTED]
  [PASS] Extracranial: partial opacification of sinuses
  [PASS] Negative: ventricles stable no hydrocephalus
  [PASS] Noise: post contrast technique note
  [PASS] Noise: per Epic note
  [PASS] Extracranial: dural sinuses
  [FAI

In [6]:
import pandas as pd
import re

MRI_CSV = '../data/mri_reports.csv'

STRUCTURAL_TYPES = [
    'MRI BRAIN',
    'MRI BRAIN WITH AND WITHOUT CONTRAST',
    'MRI BRAIN WITHOUT CONTRAST',
]

# ── Filters v4 ────────────────────────────────────────────────────────────────

BOILERPLATE_PATTERN = re.compile(
    r'forwarded to an automated communication'
    r'|findings were (?:discussed|communicated)'
    r'|alert notification of critical'
    r'|critical results were communicated'
    r'|this report has been'
    r'|electronically notify'
    r'|per epic note'
    r'|carotid stenosis reference'
    r'|(?:\[redacted\][\w\s,\.]*)?(?:md|do|phd|np|pa)[\s,]+the (?:attending|teaching) physician'
    r'|i,? the (?:attending|teaching) physician'
    r'|have reviewed the images and agree'
    r'|study (?:terminated|aborted|discontinued) (?:early|due to|at patient)'
    r'|post contrast[:\s]+following (?:iv|intravenous) administration'
    r'|following (?:iv|intravenous) administration of (?:gadolinium|contrast)'
    r'|(?:exam|study|images?|sequences?) (?:is|are) limited (?:by|due to) (?:motion|artifact|patient)'
    r'|(?:patient was not able|patient could not) (?:to )?tolerate'
    r'|only (?:diffusion|flair|dwi|t1|t2)[\w\s]+ (?:sequence|sequences|images?) (?:were|was) obtained'
    r'|(?:images?|sequences?) (?:are|were|is) degraded by'
    r'|pending postprocessing at the time of this dictation'
    r'|perfusion images? (?:are )?pending',
    re.IGNORECASE
)

HEADER_PATTERN = re.compile(
    r'^(?:brain mri|head mra|neck mra|mri brain|intracranial mra'
    r'|anterior circulation|posterior circulation|collateral circulation'
    r'|aortic\s+\[redacted\].*origin of major cervical'
    r'|impression|technique|comparison|indication|methodology'
    r'|detail|findings|neck'
    r'|ventricular system[\w\s\-]+:'
    r'|extra-axial spaces?:'
    r'|brain parenchyma:'
    r'|vascular system:'
    r'|vascular structures?:'
    r'|extracranial structures?:'
    r'|flow.related signal):?\s*$',
    re.IGNORECASE
)

NEGATIVE_PATTERN = re.compile(
    r'^(?:there (?:is|are) )?(?:no evidence of|no acute|no new|no abnormal'
    r'|no significant|unremarkable|within normal limits|no hemorrhage'
    r'|no infarct|no mass effect|no hydrocephalus|no midline shift'
    r'|no space.occupying|no enhancing lesion|no abnormal enhancement'
    r'|no restricted diffusion|no acute intracranial|no brain parenchymal mass'
    r'|no interval change)'
    r'|^the (?:major |main )?(?:intracranial )?(?:flow voids|arterial flow voids|vessels).{0,60}(?:intact|preserved|patent)'
    r'|^the (?:visualized )?(?:paranasal sinuses|mastoid air cells?|orbits|bones and extracranial|extracranial soft tissues).{0,80}(?:clear|normal|unremarkable|unchanged)'
    r'|^the (?:remaining )?ventricles? (?:are|is) (?:stable|normal|unremarkable|unchanged)'
    r'|^(?:the )?ventricles?[,\s]+sulci[,\s]+and (?:cisterns?|basal cisterns?).{0,60}(?:unremarkable|normal|stable|within normal limits)'
    r'|^ventricles?[,\s]+sulci[,\s]+and cisterns? are stable'
    r'|^(?:\[redacted\]\s+)?venous sinuses? (?:enhance|are|appear).{0,40}(?:normal|patent|intact|homogenous)'
    r'|^vascular system:.{0,120}(?:intact|preserved|patent|normal|homogenous)'
    r'|^(?:extracranial structures|paranasal sinuses|mastoid).{0,80}(?:clear|normal|unremarkable)'
    r'|^(?:post contrast|following contrast).{0,60}no (?:abnormal|significant)'
    r'|^diffusion.weighted imaging demonstrates no acute'
    r'|^asl (?:sequence )?does not show',
    re.IGNORECASE
)

EXTRACRANIAL_PATTERN = re.compile(
    r'^(?:paranasal sinuses|mastoid air cells?|orbits|skull base'
    r'|extracranial soft tissues|cervical|vertebral arter'
    r'|aortic|subclavian|carotid|nasopharynx|external carotid)'
    r'|^the (?:bones[,\s]+)?extracranial'
    r'|^the (?:bones,\s+)?extracranial soft tissues'
    r'|^the (?:visualized )?(?:paranasal sinuses|mastoid|orbits)'
    r'|^the (?:intracranial )?vertebral arteries'
    r'|^the bones,?\s+extracranial'
    r'|^(?:partial|bilateral|mild|moderate|severe|minimal)?\s*(?:opacification|thickening|mucosal).{0,40}(?:sinus|mastoid|orbit)'
    r'|(?:mastoid air cells?|middle ear cavity|maxillary sinus|sphenoid sinus|ethmoid sinus).{0,30}(?:opacif|thicken|fluid|effusion|clear|normal)'
    r'|^the (?:superior sagittal|transverse|sigmoid|straight|cavernous) sinus'
    r'|^(?:major )?(?:intracranial )?(?:dural sinus|venous sinus)'
    r'|(?:nasopharynx|pharynx).{0,60}(?:edema|mucosal|thickening|fluid)'
    r'|(?:edema|mucosal thickening).{0,40}(?:nasopharynx|pharynx)'
    r'|(?:spinal canal|neuroforaminal|neural foramen|spinal cord|c\d-\d|cervical spine|thoracic spine)'
    r'|^(?:the )?(?:intracranial vertebral arteries|basilar artery).{0,40}(?:patent|normal)'
    r'|(?:maxillary sinus|sphenoid sinus|ethmoid).{0,80}(?:asymmetr|diminutive|bowing|deviat|retention cyst)',
    re.IGNORECASE
)

def is_noise(t):
    if len(t.strip()) < 30: return True
    if HEADER_PATTERN.match(t): return True
    if BOILERPLATE_PATTERN.search(t): return True
    return False

def is_negative(t): return bool(NEGATIVE_PATTERN.match(t))
def is_extracranial(t): return bool(EXTRACRANIAL_PATTERN.search(t))

# ── Splitter ──────────────────────────────────────────────────────────────────

SECTION_SPLIT_PATTERN = re.compile(
    r'\n(?=(?:BRAIN MRI|HEAD MRA|NECK MRA|MRI BRAIN AND NECK'
    r'|SPINE|CERVICAL|THORACIC|LUMBAR|STRUCTURAL MRI|FUNCTIONAL MRI)\s*[:\n])',
    re.IGNORECASE
)
BRAIN_SECTION_PATTERN = re.compile(r'^(?:BRAIN MRI|MRI BRAIN|STRUCTURAL MRI)', re.IGNORECASE)
INLINE_HEADER_PATTERN = re.compile(r'^(?:BRAIN MRI|HEAD MRA|MRI BRAIN)\s*:?\s*\n', re.IGNORECASE)

def _extract_brain_section(text):
    if not SECTION_SPLIT_PATTERN.search(text):
        return text
    sections = SECTION_SPLIT_PATTERN.split(text)
    brain = [s for s in sections if BRAIN_SECTION_PATTERN.match(s.strip())]
    return brain[0] if brain else sections[0]

def split_report(text):
    if pd.isna(text): return []
    text = _extract_brain_section(text)
    result = []
    for p in re.split(r'\n[ \t]*\n', text):
        p = INLINE_HEADER_PATTERN.sub('', p.strip()).strip()
        if len(p) >= 30:
            result.append(p)
    return result

def split_report_old(text):
    if pd.isna(text): return []
    return [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]

# ── Load ──────────────────────────────────────────────────────────────────────
print('Loading data...')
mri = pd.read_csv(MRI_CSV, engine='python', on_bad_lines='skip')
valid = mri[mri['findings'].notna() & ~mri['findings'].str.strip().str.lower().isin(['not available','none',''])]
structural = valid[valid['Type'].isin(STRUCTURAL_TYPES)].copy()
print(f'Structural MRI rows: {len(structural):,}\n')

# ── TEST 1 ────────────────────────────────────────────────────────────────────
print('=' * 60)
print('TEST 1: Paragraph count — old vs new splitter')
print('=' * 60)
structural['old_count'] = structural['findings'].apply(lambda t: len(split_report_old(t)))
structural['new_count'] = structural['findings'].apply(lambda t: len(split_report(t)))
for label, col in [('Old', 'old_count'), ('New', 'new_count')]:
    s = structural[col]
    print(f'{label}: mean={s.mean():.2f} | median={s.median():.0f} | max={s.max()} | >20para: {(s>20).sum():,} ({100*(s>20).mean():.1f}%)')
improved = (structural['new_count'] < structural['old_count']).sum()
regressed = (structural['new_count'] > structural['old_count']).sum()
print(f'Improved: {improved:,} ({100*improved/len(structural):.1f}%) | Regressed: {regressed:,} ({100*regressed/len(structural):.1f}%)')

# ── TEST 2 ────────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 2: Multi-section report handling')
print('=' * 60)
multi = structural[structural['findings'].str.contains(r'HEAD MRA|NECK MRA|MRI BRAIN AND NECK', case=False, regex=True, na=False)]
print(f'Multi-section reports: {len(multi):,} | Old avg: {multi["old_count"].mean():.1f} | New avg: {multi["new_count"].mean():.1f}')

# ── TEST 3 ────────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 3: Filter performance')
print('=' * 60)
structural['paragraph_list'] = structural['findings'].apply(split_report)
exploded = (
    structural.explode('paragraph_list')
    .rename(columns={'paragraph_list': 'paragraph'})
    .dropna(subset=['paragraph'])
    .reset_index(drop=True)
)
noise_mask = exploded['paragraph'].apply(is_noise)
filtered = exploded[~noise_mask].copy().reset_index(drop=True)
filtered['is_negative'] = filtered['paragraph'].apply(is_negative)
filtered['is_extracranial'] = filtered['paragraph'].apply(is_extracranial)
positive = (~filtered['is_negative'] & ~filtered['is_extracranial']).sum()
print(f'Total paragraphs:            {len(exploded):,}')
print(f'Dropped (noise):             {noise_mask.sum():,} ({100*noise_mask.mean():.1f}%)')
print(f'Remaining:                   {len(filtered):,}')
print(f'  - Negative/normal:         {filtered["is_negative"].sum():,}')
print(f'  - Extracranial:            {filtered["is_extracranial"].sum():,}')
print(f'  - Positive brain findings: {positive:,}')

# ── TEST 4: Regression (fixed) ────────────────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 4: Regression check (fixed cases)')
print('=' * 60)
cases = [
    ("Extracranial: 'The bones and extracranial...'",           "The bones and extracranial soft tissues are unchanged.", 'extracranial'),
    ("Noise: physician sign-off with [REDACTED]",               "[REDACTED] MD, the teaching physician, have reviewed the images and agree with the report as written.", 'noise'),
    ("Extracranial: partial opacification of sinuses",          "Partial opacification of the maxillary sinuses bilaterally. The mastoid air cells and orbits are unremarkable.", 'extracranial'),
    ("Negative: ventricles stable no hydrocephalus",            "The remaining ventricles are stable in size and configuration. There is no hydrocephalus.", 'negative'),
    ("Noise: post contrast technique note",                     "Post contrast: Following IV administration of gadolinium chelate, no abnormal regions of enhancement are demonstrated.", 'noise'),
    ("Noise: per Epic note",                                    "Per Epic note, the patient subsequently underwent evacuation of the bifrontal fluid collection.", 'noise'),
    ("Extracranial: dural sinuses",                             "The superior sagittal sinus, internal cerebral veins, inferior sagittal sinus, straight sinus are patent.", 'extracranial'),
    ("Negative: vascular system normal",                        "Vascular system: Intracranial major dural sinus and arterial flow voids are present. Venous sinuses are homogenous post contrast.", 'negative'),
    ("Extracranial: mastoid air cells mid-sentence",            "Patient status post left temporal craniotomy. Again opacification of left middle ear cavity and mastoid air cells.", 'extracranial'),
    ("Negative: ventricles sulci cisterns unremarkable",        "The ventricles, sulci, and basal cisterns are otherwise within normal limits.", 'negative'),
    ("Negative: ventricles sulci cisterns stable",              "Ventricles, sulci, and cisterns are stable in size and configuration.", 'negative'),
    ("Negative: venous sinuses enhance normally",               "[REDACTED] venous sinuses enhance normally.", 'negative'),
    ("Extracranial: nasopharynx mucosal thickening",            "There is mild edema and mucosal thickening in the nasopharynx anteriorly.", 'extracranial'),
    ("Extracranial: maxillary sinus asymmetry",                 "Again seen is an asymmetrically diminutive appearance of the right maxillary sinus with lateral bowing.", 'extracranial'),
    ("Noise: exam limited by motion artifact",                  "The axial images are limited due to patient motion artifact. Within this limitation, no high-grade spinal canal stenosis.", 'noise'),
    ("Noise: patient could not tolerate",                       "Only diffusion and FLAIR sequences were obtained as the patient was not able to tolerate further imaging.", 'noise'),
    ("Noise: perfusion pending",                                "Of note, perfusion images are pending postprocessing at the time of this dictation.", 'noise'),
    ("Extracranial: spinal canal",                              "There is a masslike lesion centered along the right lateral spinal canal at the C1-2 level.", 'extracranial'),
    ("Extracranial: sinus content mid-paragraph",               "Minimal scattered mucosal thickening involving the visualized paranasal sinuses. No orbital abnormality. The bones and extracranial soft tissues are otherwise unremarkable.", 'extracranial'),
    ("Extracranial: external carotid arteries",                 "External Carotid Arteries: Normal. No stenosis, occlusion or dissection.", 'extracranial'),
]

all_passed = True
for desc, text, expected in cases:
    if expected == 'noise':          result = is_noise(text)
    elif expected == 'negative':     result = is_negative(text)
    elif expected == 'extracranial': result = is_extracranial(text)
    status = 'PASS' if result else 'FAIL'
    if not result: all_passed = False
    print(f'  [{status}] {desc}')
print(f'\n{"All regression tests passed!" if all_passed else "Some tests FAILED -- fix patterns before updating repo."}')

# ── TEST 5: Random positive findings ─────────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 5: 40 random positive brain findings (randomized each run)')
print('=' * 60)
pos = filtered[~filtered['is_negative'] & ~filtered['is_extracranial']]
for i, (_, row) in enumerate(pos.sample(40).iterrows()):
    print(f'\n[{i+1}] {row["paragraph"][:200]}')

print('\n' + '=' * 60)
print('ALL TESTS COMPLETE')
print('=' * 60)

Loading data...
Structural MRI rows: 36,924

TEST 1: Paragraph count — old vs new splitter
Old: mean=9.05 | median=8 | max=88 | >20para: 1,475 (4.0%)
New: mean=7.94 | median=7 | max=70 | >20para: 483 (1.3%)
Improved: 11,216 (30.4%) | Regressed: 0 (0.0%)

TEST 2: Multi-section report handling
Multi-section reports: 1,725 | Old avg: 19.6 | New avg: 7.0

TEST 3: Filter performance
Total paragraphs:            293,309
Dropped (noise):             8,540 (2.9%)
Remaining:                   284,769
  - Negative/normal:         64,791
  - Extracranial:            26,057
  - Positive brain findings: 201,145

TEST 4: Regression check (fixed cases)
  [FAIL] Extracranial: 'The bones and extracranial...'
  [PASS] Noise: physician sign-off with [REDACTED]
  [PASS] Extracranial: partial opacification of sinuses
  [PASS] Negative: ventricles stable no hydrocephalus
  [PASS] Noise: post contrast technique note
  [PASS] Noise: per Epic note
  [PASS] Extracranial: dural sinuses
  [PASS] Negative: vascula

In [7]:
import pandas as pd
import re

MRI_CSV = '../data/mri_reports.csv'

STRUCTURAL_TYPES = [
    'MRI BRAIN',
    'MRI BRAIN WITH AND WITHOUT CONTRAST',
    'MRI BRAIN WITHOUT CONTRAST',
]

# ── Filters v5 ────────────────────────────────────────────────────────────────

BOILERPLATE_PATTERN = re.compile(
    r'forwarded to an automated communication'
    r'|findings were (?:discussed|communicated)'
    r'|alert notification of critical'
    r'|critical results were communicated'
    r'|this report has been'
    r'|electronically notify'
    r'|per epic note'
    r'|carotid stenosis reference'
    r'|(?:\[redacted\][\w\s,\.]*)?(?:md|do|phd|np|pa)[\s,]+the (?:attending|teaching) physician'
    r'|i,? the (?:attending|teaching) physician'
    r'|have reviewed the images and agree'
    r'|study (?:terminated|aborted|discontinued) (?:early|due to|at patient)'
    r'|post contrast[:\s]+following (?:iv|intravenous) administration'
    r'|following (?:iv|intravenous) administration of (?:gadolinium|contrast)'
    r'|(?:exam|study|images?|sequences?) (?:is|are) limited (?:by|due to) (?:motion|artifact|patient)'
    r'|(?:patient was not able|patient could not) (?:to )?tolerate'
    r'|(?:patient was )?removed from the magnet'
    r'|only (?:diffusion|flair|dwi|t1|t2)[\w\s]+ (?:sequence|sequences|images?) (?:were|was) obtained'
    r'|(?:images?|sequences?) (?:are|were|is) degraded by'
    r'|pending postprocessing at the time of this dictation'
    r'|perfusion images? (?:are )?pending'
    r'|study quality is limited'
    r'|there is motion artifact'
    r'|(?:asl|dwi) sequence is nondiagnostic'
    r'|you can find out more about our efforts'
    r'|visiting \[redacted\]',
    re.IGNORECASE
)

HEADER_PATTERN = re.compile(
    r'^(?:brain mri|head mra|neck mra|mri brain|intracranial mra'
    r'|anterior circulation|posterior circulation|collateral circulation'
    r'|aortic\s+\[redacted\].*origin of major cervical'
    r'|impression|technique|comparison|indication|methodology'
    r'|detail|findings|neck'
    r'|ventricular system[\w\s\-]+:'
    r'|extra-axial spaces?:'
    r'|brain parenchyma:'
    r'|vascular system:'
    r'|vascular structures?:'
    r'|extracranial structures?:?'
    r'|flow.related signal'
    r'|synopsis for clinical management'
    r'|miscellaneous:'
    r'|calvarium:'
    r'|anterior circulation:):?\s*$',
    re.IGNORECASE
)

NEGATIVE_PATTERN = re.compile(
    r'^(?:there (?:is|are) )?(?:no evidence of|no acute|no new|no abnormal'
    r'|no significant|unremarkable|within normal limits|no hemorrhage'
    r'|no infarct|no mass effect|no hydrocephalus|no midline shift'
    r'|no space.occupying|no enhancing lesion|no abnormal enhancement'
    r'|no restricted diffusion|no acute intracranial|no brain parenchymal mass'
    r'|no interval change|no intracranial mass)'
    r'|^the (?:major |main )?(?:intracranial )?(?:flow voids|arterial flow voids|vessels).{0,60}(?:intact|preserved|patent)'
    r'|^the (?:visualized )?(?:paranasal sinuses|mastoid air cells?|orbits|bones and extracranial|extracranial soft tissues).{0,80}(?:clear|normal|unremarkable|unchanged)'
    r'|^the (?:remaining )?ventricles? (?:are|is) (?:stable|normal|unremarkable|unchanged)'
    r'|^(?:the )?ventricles?[,\s]+sulci[,\s]+and (?:cisterns?|basal cisterns?).{0,60}(?:unremarkable|normal|stable|within normal limits)'
    r'|^ventricles?[,\s]+sulci[,\s]+and cisterns? are stable'
    r'|^there has been no (?:significant )?interval change in (?:size|the size)'
    r'|^(?:\[redacted\]\s+)?venous sinuses? (?:enhance|are|appear).{0,40}(?:normal|patent|intact|homogenous)'
    r'|^vascular system:.{0,120}(?:intact|preserved|patent|normal|homogenous)'
    r'|^(?:extracranial structures|paranasal sinuses|mastoid).{0,80}(?:clear|normal|unremarkable)'
    r'|^(?:post contrast|following contrast).{0,60}no (?:abnormal|significant)'
    r'|^(?:there are )?no regions of abnormal restricted diffusivity'
    r'|^diffusion.weighted imaging demonstrates no acute'
    r'|^asl (?:sequence )?does not show'
    r'|^calvarium:.{0,60}(?:normal|within normal limits|unremarkable)',
    re.IGNORECASE
)

EXTRACRANIAL_PATTERN = re.compile(
    r'^(?:paranasal sinuses|mastoid air cells?|orbits|skull base'
    r'|extracranial soft tissues|cervical|vertebral arter'
    r'|aortic|subclavian|carotid|nasopharynx|external carotid'
    r'|mra\s+\[?redacted\]?)'
    r'|^the bones'
    r'|^the (?:visualized )?(?:paranasal sinuses|mastoid|orbits)'
    r'|^the (?:intracranial )?vertebral arteries'
    r'|^(?:partial|bilateral|mild|moderate|severe|minimal|scattered)?\s*(?:opacification|thickening|mucosal).{0,40}(?:sinus|mastoid|orbit)'
    r'|(?:mastoid air cells?|middle ear cav|maxillary sinus|sphenoid sinus|ethmoid sinus|paranasal sinus).{0,50}(?:opacif|thicken|fluid|effusion|clear|normal|air cell)'
    r'|(?:superior sagittal|transverse|sigmoid|straight sinus|cavernous sinus|dural (?:venous )?sinus|venous sinus).{0,60}(?:patent|thrombus|laceration|signal|enhance)'
    r'|(?:nasopharynx|pharynx).{0,60}(?:edema|mucosal|thickening|fluid)'
    r'|(?:edema|mucosal thickening).{0,40}(?:nasopharynx|pharynx)'
    r'|(?:spinal canal|neuroforaminal|neural foramen|spinal cord|cauda equina|conus terminates|c\d-\d|l\d[ -]|cervical spine|thoracic spine|cervicomedullary)'
    r'|^(?:the )?(?:intracranial vertebral arteries|basilar artery).{0,40}(?:patent|normal)'
    r'|^(?:left|right)? vertebral artery (?:demonstrates|shows)'
    r'|^anterior circulation:'
    r'|^(?:no )?hemodynamically.significant stenosis'
    r'|(?:maxillary sinus|sphenoid sinus|ethmoid).{0,80}(?:asymmetr|diminutive|bowing|deviat|retention cyst)',
    re.IGNORECASE
)

INLINE_SUBSECTION_PATTERN = re.compile(
    r'^(?:brain parenchyma|ventricular system[\w\s\-]*|extra-axial spaces?'
    r'|vascular system|extracranial structures?|calvarium|miscellaneous'
    r'|synopsis for clinical management)\s*:\s*',
    re.IGNORECASE
)

def strip_inline_header(text):
    return INLINE_SUBSECTION_PATTERN.sub('', text).strip()

def is_noise(t):
    if len(t.strip()) < 30: return True
    if HEADER_PATTERN.match(t): return True
    if BOILERPLATE_PATTERN.search(t): return True
    return False

def is_negative(t): return bool(NEGATIVE_PATTERN.match(t))
def is_extracranial(t): return bool(EXTRACRANIAL_PATTERN.search(t))

# ── Splitter ──────────────────────────────────────────────────────────────────

SECTION_SPLIT_PATTERN = re.compile(
    r'\n(?=(?:BRAIN MRI|HEAD MRA|NECK MRA|MRI BRAIN AND NECK'
    r'|SPINE|CERVICAL|THORACIC|LUMBAR|STRUCTURAL MRI|FUNCTIONAL MRI)\s*[:\n])',
    re.IGNORECASE
)
BRAIN_SECTION_PATTERN = re.compile(r'^(?:BRAIN MRI|MRI BRAIN|STRUCTURAL MRI)', re.IGNORECASE)
INLINE_HEADER_PATTERN = re.compile(r'^(?:BRAIN MRI|HEAD MRA|MRI BRAIN)\s*:?\s*\n', re.IGNORECASE)

def _extract_brain_section(text):
    if not SECTION_SPLIT_PATTERN.search(text):
        return text
    sections = SECTION_SPLIT_PATTERN.split(text)
    brain = [s for s in sections if BRAIN_SECTION_PATTERN.match(s.strip())]
    return brain[0] if brain else sections[0]

def split_report(text):
    if pd.isna(text): return []
    text = _extract_brain_section(text)
    result = []
    for p in re.split(r'\n[ \t]*\n', text):
        p = INLINE_HEADER_PATTERN.sub('', p.strip()).strip()
        p = strip_inline_header(p)
        if len(p) >= 30:
            result.append(p)
    return result

def split_report_old(text):
    if pd.isna(text): return []
    return [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]

# ── Load ──────────────────────────────────────────────────────────────────────
print('Loading data...')
mri = pd.read_csv(MRI_CSV, engine='python', on_bad_lines='skip')
valid = mri[mri['findings'].notna() & ~mri['findings'].str.strip().str.lower().isin(['not available','none',''])]
structural = valid[valid['Type'].isin(STRUCTURAL_TYPES)].copy()
print(f'Structural MRI rows: {len(structural):,}\n')

# ── TEST 1 ────────────────────────────────────────────────────────────────────
print('=' * 60)
print('TEST 1: Paragraph count — old vs new splitter')
print('=' * 60)
structural['old_count'] = structural['findings'].apply(lambda t: len(split_report_old(t)))
structural['new_count'] = structural['findings'].apply(lambda t: len(split_report(t)))
for label, col in [('Old', 'old_count'), ('New', 'new_count')]:
    s = structural[col]
    print(f'{label}: mean={s.mean():.2f} | median={s.median():.0f} | max={s.max()} | >20para: {(s>20).sum():,} ({100*(s>20).mean():.1f}%)')
improved = (structural['new_count'] < structural['old_count']).sum()
regressed = (structural['new_count'] > structural['old_count']).sum()
print(f'Improved: {improved:,} ({100*improved/len(structural):.1f}%) | Regressed: {regressed:,} ({100*regressed/len(structural):.1f}%)')

# ── TEST 2 ────────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 2: Multi-section report handling')
print('=' * 60)
multi = structural[structural['findings'].str.contains(r'HEAD MRA|NECK MRA|MRI BRAIN AND NECK', case=False, regex=True, na=False)]
print(f'Multi-section reports: {len(multi):,} | Old avg: {multi["old_count"].mean():.1f} | New avg: {multi["new_count"].mean():.1f}')

# ── TEST 3 ────────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 3: Filter performance')
print('=' * 60)
structural['paragraph_list'] = structural['findings'].apply(split_report)
exploded = (
    structural.explode('paragraph_list')
    .rename(columns={'paragraph_list': 'paragraph'})
    .dropna(subset=['paragraph'])
    .reset_index(drop=True)
)
noise_mask = exploded['paragraph'].apply(is_noise)
filtered = exploded[~noise_mask].copy().reset_index(drop=True)
filtered['is_negative'] = filtered['paragraph'].apply(is_negative)
filtered['is_extracranial'] = filtered['paragraph'].apply(is_extracranial)
positive = (~filtered['is_negative'] & ~filtered['is_extracranial']).sum()
print(f'Total paragraphs:            {len(exploded):,}')
print(f'Dropped (noise):             {noise_mask.sum():,} ({100*noise_mask.mean():.1f}%)')
print(f'Remaining:                   {len(filtered):,}')
print(f'  - Negative/normal:         {filtered["is_negative"].sum():,}')
print(f'  - Extracranial:            {filtered["is_extracranial"].sum():,}')
print(f'  - Positive brain findings: {positive:,}')

# ── TEST 4 ────────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 4: Regression check (fixed cases)')
print('=' * 60)
cases = [
    ("Extracranial: 'The bones and extracranial...'",           "The bones and extracranial soft tissues are unchanged.", 'extracranial'),
    ("Noise: physician sign-off with [REDACTED]",               "[REDACTED] MD, the teaching physician, have reviewed the images and agree with the report as written.", 'noise'),
    ("Extracranial: partial opacification of sinuses",          "Partial opacification of the maxillary sinuses bilaterally. The mastoid air cells and orbits are unremarkable.", 'extracranial'),
    ("Negative: ventricles stable no hydrocephalus",            "The remaining ventricles are stable in size and configuration. There is no hydrocephalus.", 'negative'),
    ("Noise: post contrast technique note",                     "Post contrast: Following IV administration of gadolinium chelate, no abnormal regions of enhancement are demonstrated.", 'noise'),
    ("Noise: per Epic note",                                    "Per Epic note, the patient subsequently underwent evacuation of the bifrontal fluid collection.", 'noise'),
    ("Extracranial: dural sinuses",                             "The superior sagittal sinus, internal cerebral veins, inferior sagittal sinus, straight sinus are patent.", 'extracranial'),
    ("Negative: vascular system normal",                        "Vascular system: Intracranial major dural sinus and arterial flow voids are present. Venous sinuses are homogenous post contrast.", 'negative'),
    ("Extracranial: mastoid air cells mid-sentence",            "Patient status post left temporal craniotomy. Again opacification of left middle ear cavity and mastoid air cells.", 'extracranial'),
    ("Negative: ventricles sulci cisterns unremarkable",        "The ventricles, sulci, and basal cisterns are otherwise within normal limits.", 'negative'),
    ("Negative: ventricles sulci cisterns stable",              "Ventricles, sulci, and cisterns are stable in size and configuration.", 'negative'),
    ("Negative: venous sinuses enhance normally",               "[REDACTED] venous sinuses enhance normally.", 'negative'),
    ("Extracranial: nasopharynx mucosal thickening",            "There is mild edema and mucosal thickening in the nasopharynx anteriorly.", 'extracranial'),
    ("Extracranial: maxillary sinus asymmetry",                 "Again seen is an asymmetrically diminutive appearance of the right maxillary sinus with lateral bowing.", 'extracranial'),
    ("Noise: exam limited by motion artifact",                  "The axial images are limited due to patient motion artifact. Within this limitation, no high-grade spinal canal stenosis.", 'noise'),
    ("Noise: patient could not tolerate",                       "Only diffusion and FLAIR sequences were obtained as the patient was not able to tolerate further imaging.", 'noise'),
    ("Noise: perfusion pending",                                "Of note, perfusion images are pending postprocessing at the time of this dictation.", 'noise'),
    ("Extracranial: spinal canal",                              "There is a masslike lesion centered along the right lateral spinal canal at the C1-2 level.", 'extracranial'),
    ("Extracranial: sinus content mid-paragraph",               "Minimal scattered mucosal thickening involving the visualized paranasal sinuses. No orbital abnormality. The bones and extracranial soft tissues are otherwise unremarkable.", 'extracranial'),
    ("Extracranial: external carotid arteries",                 "External Carotid Arteries: Normal. No stenosis, occlusion or dissection.", 'extracranial'),
    ("Noise: study quality limited",                            "Study quality is limited secondary to motion artifact.", 'noise'),
    ("Noise: motion artifact",                                  "There is motion artifact, somewhat limiting evaluation.", 'noise'),
    ("Noise: ASL nondiagnostic",                                "The ASL sequence is nondiagnostic and degraded by artifact.", 'noise'),
    ("Noise: removed from magnet",                              "The patient was removed from the magnet prematurely before the completion of the majority of the examination.", 'noise'),
    ("Noise: website URL",                                      "You can find out more about our efforts to improve the [REDACTED] of radiology reports by visiting [REDACTED].", 'noise'),
    ("Extracranial: MRA finding",                               "MRA [REDACTED]: No hemodynamically-significant stenosis of the innominate or bilateral common carotid arteries.", 'extracranial'),
    ("Extracranial: cauda equina / spine",                      "The conus terminates at the level of L1, in normal anatomic position. The cauda equina are normal.", 'extracranial'),
    ("Extracranial: vertebral artery dissection",               "Left vertebral artery demonstrates multiple segments of narrowing and dilatation, highly suspicious for a vertebral artery dissection.", 'extracranial'),
    ("Extracranial: sagittal sinus laceration",                 "Loss of signal in the anterior portion of the superior sagittal sinus secondary to known laceration injury.", 'extracranial'),
    ("Negative: no regions of abnormal restricted diffusivity", "There are no regions of abnormal restricted diffusivity demonstrated on the DWI.", 'negative'),
    ("Inline header stripped: Brain Parenchyma:",               "Brain Parenchyma: Interval decrease in extent of T2/FLAIR abnormality surrounding the left basal ganglia resection cavity.", 'strip_check'),
]

all_passed = True
for desc, text, expected in cases:
    if expected == 'strip_check':
        stripped = strip_inline_header(text)
        result = not stripped.startswith('Brain Parenchyma')
        label = 'header stripped'
    elif expected == 'noise':          result = is_noise(text)
    elif expected == 'negative':       result = is_negative(text)
    elif expected == 'extracranial':   result = is_extracranial(text)
    status = 'PASS' if result else 'FAIL'
    if not result: all_passed = False
    print(f'  [{status}] {desc}')
print(f'\n{"All regression tests passed!" if all_passed else "Some tests FAILED -- fix patterns before updating repo."}')

# ── TEST 5 ────────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 5: 40 random positive brain findings (randomized each run)')
print('=' * 60)
pos = filtered[~filtered['is_negative'] & ~filtered['is_extracranial']]
for i, (_, row) in enumerate(pos.sample(40).iterrows()):
    print(f'\n[{i+1}] {row["paragraph"][:200]}')

print('\n' + '=' * 60)
print('ALL TESTS COMPLETE')
print('=' * 60)

Loading data...
Structural MRI rows: 36,924

TEST 1: Paragraph count — old vs new splitter
Old: mean=9.05 | median=8 | max=88 | >20para: 1,475 (4.0%)
New: mean=7.87 | median=7 | max=70 | >20para: 481 (1.3%)
Improved: 11,848 (32.1%) | Regressed: 0 (0.0%)

TEST 2: Multi-section report handling
Multi-section reports: 1,725 | Old avg: 19.6 | New avg: 6.9

TEST 3: Filter performance
Total paragraphs:            290,717
Dropped (noise):             8,666 (3.0%)
Remaining:                   282,051
  - Negative/normal:         68,069
  - Extracranial:            37,270
  - Positive brain findings: 189,746

TEST 4: Regression check (fixed cases)
  [PASS] Extracranial: 'The bones and extracranial...'
  [PASS] Noise: physician sign-off with [REDACTED]
  [PASS] Extracranial: partial opacification of sinuses
  [PASS] Negative: ventricles stable no hydrocephalus
  [PASS] Noise: post contrast technique note
  [PASS] Noise: per Epic note
  [PASS] Extracranial: dural sinuses
  [PASS] Negative: vascula

In [8]:
import pandas as pd
import re

MRI_CSV = '../data/mri_reports.csv'

STRUCTURAL_TYPES = [
    'MRI BRAIN',
    'MRI BRAIN WITH AND WITHOUT CONTRAST',
    'MRI BRAIN WITHOUT CONTRAST',
]

# ── Filters v6 ────────────────────────────────────────────────────────────────

BOILERPLATE_PATTERN = re.compile(
    r'forwarded to an automated communication'
    r'|findings were (?:discussed|communicated)'
    r'|alert notification of critical'
    r'|critical results were communicated'
    r'|this report has been'
    r'|electronically notify'
    r'|per epic note'
    r'|carotid stenosis reference'
    r'|(?:\[redacted\][\w\s,\.]*)?(?:md|do|phd|np|pa)[\s,]+the (?:attending|teaching) physician'
    r'|i,? the (?:attending|teaching) physician'
    r'|have reviewed the images and agree'
    r'|study (?:terminated|aborted|discontinued) (?:early|due to|at patient)'
    r'|post contrast[:\s]+following (?:iv|intravenous) administration'
    r'|following (?:iv|intravenous) administration of (?:gadolinium|contrast)'
    r'|(?:exam|study|images?|sequences?|imaging quality) (?:is|are) (?:limited|degraded) (?:by|due to|secondary to) (?:motion|artifact|patient)'
    r'|(?:patient was not able|patient could not) (?:to )?tolerate'
    r'|(?:patient was )?removed from the magnet'
    r'|only (?:diffusion|flair|dwi|t1|t2)[\w\s]+ (?:sequence|sequences|images?) (?:were|was) obtained'
    r'|(?:images?|sequences?) (?:are|were|is) degraded by'
    r'|pending postprocessing at the time of this dictation'
    r'|perfusion images? (?:are )?pending'
    r'|study quality is limited'
    r'|there is motion artifact'
    r'|(?:asl|dwi) sequence is nondiagnostic'
    r'|you can find out more about our efforts'
    r'|visiting \[redacted\]'
    r'|suboptimal evaluation (?:for|of|secondary to).{0,60}(?:motion|artifact|technique)',
    re.IGNORECASE
)

HEADER_PATTERN = re.compile(
    r'^(?:brain mri|head mra|neck mra|mri brain|intracranial mra'
    r'|anterior circulation|posterior circulation|collateral circulation'
    r'|aortic\s+\[redacted\].*origin of major cervical'
    r'|impression|technique|comparison|indication|methodology'
    r'|detail|findings|neck'
    r'|ventricular system[\w\s\-]+:'
    r'|extra-axial spaces?:'
    r'|brain parenchyma:'
    r'|vascular system:'
    r'|vascular structures?:'
    r'|extracranial structures?:?'
    r'|flow.related signal'
    r'|synopsis for clinical management'
    r'|miscellaneous:'
    r'|calvarium:'
    r'|anterior circulation:):?\s*$',
    re.IGNORECASE
)

NEGATIVE_PATTERN = re.compile(
    r'^(?:there (?:is|are) )?(?:no evidence of|no acute|no new|no abnormal'
    r'|no significant|unremarkable|within normal limits|no hemorrhage'
    r'|no infarct|no mass effect|no hydrocephalus|no midline shift'
    r'|no space.occupying|no enhancing lesion|no abnormal enhancement'
    r'|no restricted diffusion|no acute intracranial|no brain parenchymal mass'
    r'|no interval change|no intracranial mass)'
    r'|^the (?:major |main )?(?:intracranial )?(?:flow voids|arterial flow voids|vessels).{0,60}(?:intact|preserved|patent|maintained)'
    r'|^the (?:visualized )?(?:paranasal sinuses|mastoid air cells?|orbits|bones and extracranial|extracranial soft tissues).{0,80}(?:clear|normal|unremarkable|unchanged)'
    r'|^the (?:remaining )?ventricles? (?:are|is) (?:stable|normal|unremarkable|unchanged)'
    r'|^(?:the )?ventricles?[,\s]+sulci[,\s]+and (?:cisterns?|basal cisterns?).{0,60}(?:unremarkable|normal|stable|within normal limits)'
    r'|^ventricles?[,\s]+sulci[,\s]+and cisterns? are stable'
    r'|^there has been no (?:significant )?interval change in (?:size|the size)'
    r'|^(?:\[redacted\]\s+)?venous sinuses? (?:enhance|are|appear).{0,40}(?:normal|patent|intact|homogenous)'
    r'|^vascular system:.{0,120}(?:intact|preserved|patent|normal|homogenous)'
    r'|^(?:extracranial structures|paranasal sinuses|mastoid).{0,80}(?:clear|normal|unremarkable)'
    r'|^(?:otherwise,?\s+)?the bones(?: and extracranial soft tissues)?.{0,60}(?:unremarkable|normal|unchanged)'
    r'|^(?:post contrast|following contrast).{0,60}no (?:abnormal|significant)'
    r'|^(?:there are )?no regions of abnormal restricted diffusivity'
    r'|^diffusion.weighted imaging demonstrates no acute'
    r'|^asl (?:sequence )?does not show'
    r'|^calvarium:.{0,60}(?:normal|within normal limits|unremarkable)',
    re.IGNORECASE
)

EXTRACRANIAL_PATTERN = re.compile(
    r'^(?:paranasal sinuses|mastoid air cells?|orbits|skull base'
    r'|extracranial soft tissues|cervical|vertebral arter'
    r'|aortic|subclavian|carotid|nasopharynx|external carotid'
    r'|mra\s+\[?redacted\]?)'
    r'|^(?:otherwise,?\s+)?the bones'
    r'|^the (?:visualized )?(?:paranasal sinuses|mastoid|orbits)'
    r'|^the (?:intracranial )?vertebral arteries'
    r'|^(?:partial|bilateral|mild|moderate|severe|minimal|scattered)?\s*(?:opacification|thickening|mucosal).{0,40}(?:sinus|mastoid|orbit)'
    r'|(?:mastoid air cells?|middle ear cav|maxillary sinus|sphenoid sinus|ethmoid sinus|paranasal sinus).{0,50}(?:opacif|thicken|fluid|effusion|clear|normal|air cell)'
    r'|(?:superior sagittal|transverse|sigmoid|straight sinus|cavernous sinus|dural (?:venous )?sinus|venous sinus).{0,60}(?:patent|thrombus|laceration|signal|enhance)'
    r'|(?:nasopharynx|pharynx).{0,60}(?:edema|mucosal|thickening|fluid)'
    r'|(?:edema|mucosal thickening).{0,40}(?:nasopharynx|pharynx)'
    r'|(?:spinal canal|neuroforaminal|neural foramen|spinal cord|cauda equina|conus terminates|c\d-\d|l\d[ -]|cervical spine|thoracic spine|cervicomedullary)'
    r'|^(?:the )?(?:intracranial vertebral arteries|basilar artery).{0,40}(?:patent|normal)'
    r'|^(?:left|right)? vertebral artery (?:demonstrates|shows)'
    r'|^anterior circulation:'
    r'|^(?:no )?hemodynamically.significant stenosis'
    r'|(?:aortic arch|three.vessel arch|subclavian arter|vertebral arter(?:ies)? arise)'
    r'|(?:parietal bone|frontal bone|occipital bone|temporal bone|calvarium|calvarial).{0,60}(?:focus|lesion|metastas|hyperintens|signal)'
    r'|(?:maxillary sinus|sphenoid sinus|ethmoid).{0,80}(?:asymmetr|diminutive|bowing|deviat|retention cyst)',
    re.IGNORECASE
)

INLINE_SUBSECTION_PATTERN = re.compile(
    r'^(?:brain parenchyma|ventricular system[\w\s\-]*|extra-axial spaces?'
    r'|vascular system|extracranial structures?|calvarium|miscellaneous'
    r'|synopsis for clinical management)\s*:\s*',
    re.IGNORECASE
)

def strip_inline_header(text):
    return INLINE_SUBSECTION_PATTERN.sub('', text).strip()

def is_noise(t):
    if len(t.strip()) < 30: return True
    if HEADER_PATTERN.match(t): return True
    if BOILERPLATE_PATTERN.search(t): return True
    return False

def is_negative(t): return bool(NEGATIVE_PATTERN.match(t))
def is_extracranial(t): return bool(EXTRACRANIAL_PATTERN.search(t))

# ── Splitter ──────────────────────────────────────────────────────────────────

SECTION_SPLIT_PATTERN = re.compile(
    r'\n(?=(?:BRAIN MRI|HEAD MRA|NECK MRA|MRI BRAIN AND NECK'
    r'|SPINE|CERVICAL|THORACIC|LUMBAR|STRUCTURAL MRI|FUNCTIONAL MRI)\s*[:\n])',
    re.IGNORECASE
)
BRAIN_SECTION_PATTERN = re.compile(r'^(?:BRAIN MRI|MRI BRAIN|STRUCTURAL MRI)', re.IGNORECASE)
INLINE_HEADER_PATTERN = re.compile(r'^(?:BRAIN MRI|HEAD MRA|MRI BRAIN)\s*:?\s*\n', re.IGNORECASE)

def _extract_brain_section(text):
    if not SECTION_SPLIT_PATTERN.search(text):
        return text
    sections = SECTION_SPLIT_PATTERN.split(text)
    brain = [s for s in sections if BRAIN_SECTION_PATTERN.match(s.strip())]
    return brain[0] if brain else sections[0]

def split_report(text):
    if pd.isna(text): return []
    text = _extract_brain_section(text)
    result = []
    for p in re.split(r'\n[ \t]*\n', text):
        p = INLINE_HEADER_PATTERN.sub('', p.strip()).strip()
        p = strip_inline_header(p)
        if len(p) >= 30:
            result.append(p)
    return result

def split_report_old(text):
    if pd.isna(text): return []
    return [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]

# ── Load ──────────────────────────────────────────────────────────────────────
print('Loading data...')
mri = pd.read_csv(MRI_CSV, engine='python', on_bad_lines='skip')
valid = mri[mri['findings'].notna() & ~mri['findings'].str.strip().str.lower().isin(['not available','none',''])]
structural = valid[valid['Type'].isin(STRUCTURAL_TYPES)].copy()
print(f'Structural MRI rows: {len(structural):,}\n')

# ── TEST 1 ────────────────────────────────────────────────────────────────────
print('=' * 60)
print('TEST 1: Paragraph count — old vs new splitter')
print('=' * 60)
structural['old_count'] = structural['findings'].apply(lambda t: len(split_report_old(t)))
structural['new_count'] = structural['findings'].apply(lambda t: len(split_report(t)))
for label, col in [('Old', 'old_count'), ('New', 'new_count')]:
    s = structural[col]
    print(f'{label}: mean={s.mean():.2f} | median={s.median():.0f} | max={s.max()} | >20para: {(s>20).sum():,} ({100*(s>20).mean():.1f}%)')
improved = (structural['new_count'] < structural['old_count']).sum()
regressed = (structural['new_count'] > structural['old_count']).sum()
print(f'Improved: {improved:,} ({100*improved/len(structural):.1f}%) | Regressed: {regressed:,} ({100*regressed/len(structural):.1f}%)')

# ── TEST 2 ────────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 2: Multi-section report handling')
print('=' * 60)
multi = structural[structural['findings'].str.contains(r'HEAD MRA|NECK MRA|MRI BRAIN AND NECK', case=False, regex=True, na=False)]
print(f'Multi-section reports: {len(multi):,} | Old avg: {multi["old_count"].mean():.1f} | New avg: {multi["new_count"].mean():.1f}')

# ── TEST 3 ────────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 3: Filter performance')
print('=' * 60)
structural['paragraph_list'] = structural['findings'].apply(split_report)
exploded = (
    structural.explode('paragraph_list')
    .rename(columns={'paragraph_list': 'paragraph'})
    .dropna(subset=['paragraph'])
    .reset_index(drop=True)
)
noise_mask = exploded['paragraph'].apply(is_noise)
filtered = exploded[~noise_mask].copy().reset_index(drop=True)
filtered['is_negative'] = filtered['paragraph'].apply(is_negative)
filtered['is_extracranial'] = filtered['paragraph'].apply(is_extracranial)
positive = (~filtered['is_negative'] & ~filtered['is_extracranial']).sum()
print(f'Total paragraphs:            {len(exploded):,}')
print(f'Dropped (noise):             {noise_mask.sum():,} ({100*noise_mask.mean():.1f}%)')
print(f'Remaining:                   {len(filtered):,}')
print(f'  - Negative/normal:         {filtered["is_negative"].sum():,}')
print(f'  - Extracranial:            {filtered["is_extracranial"].sum():,}')
print(f'  - Positive brain findings: {positive:,}')

# ── TEST 4 ────────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 4: Regression check (fixed cases)')
print('=' * 60)
cases = [
    ("Extracranial: 'The bones and extracranial...'",               "The bones and extracranial soft tissues are unchanged.", 'extracranial'),
    ("Noise: physician sign-off with [REDACTED]",                   "[REDACTED] MD, the teaching physician, have reviewed the images and agree with the report as written.", 'noise'),
    ("Extracranial: partial opacification of sinuses",              "Partial opacification of the maxillary sinuses bilaterally. The mastoid air cells and orbits are unremarkable.", 'extracranial'),
    ("Negative: ventricles stable no hydrocephalus",                "The remaining ventricles are stable in size and configuration. There is no hydrocephalus.", 'negative'),
    ("Noise: post contrast technique note",                         "Post contrast: Following IV administration of gadolinium chelate, no abnormal regions of enhancement are demonstrated.", 'noise'),
    ("Noise: per Epic note",                                        "Per Epic note, the patient subsequently underwent evacuation of the bifrontal fluid collection.", 'noise'),
    ("Extracranial: dural sinuses",                                 "The superior sagittal sinus, internal cerebral veins, inferior sagittal sinus, straight sinus are patent.", 'extracranial'),
    ("Negative: vascular system normal",                            "Vascular system: Intracranial major dural sinus and arterial flow voids are present. Venous sinuses are homogenous post contrast.", 'negative'),
    ("Extracranial: mastoid air cells mid-sentence",                "Patient status post left temporal craniotomy. Again opacification of left middle ear cavity and mastoid air cells.", 'extracranial'),
    ("Negative: ventricles sulci cisterns unremarkable",            "The ventricles, sulci, and basal cisterns are otherwise within normal limits.", 'negative'),
    ("Negative: ventricles sulci cisterns stable",                  "Ventricles, sulci, and cisterns are stable in size and configuration.", 'negative'),
    ("Negative: venous sinuses enhance normally",                   "[REDACTED] venous sinuses enhance normally.", 'negative'),
    ("Extracranial: nasopharynx mucosal thickening",                "There is mild edema and mucosal thickening in the nasopharynx anteriorly.", 'extracranial'),
    ("Extracranial: maxillary sinus asymmetry",                     "Again seen is an asymmetrically diminutive appearance of the right maxillary sinus with lateral bowing.", 'extracranial'),
    ("Noise: exam limited by motion artifact",                      "The axial images are limited due to patient motion artifact. Within this limitation, no high-grade spinal canal stenosis.", 'noise'),
    ("Noise: patient could not tolerate",                           "Only diffusion and FLAIR sequences were obtained as the patient was not able to tolerate further imaging.", 'noise'),
    ("Noise: perfusion pending",                                    "Of note, perfusion images are pending postprocessing at the time of this dictation.", 'noise'),
    ("Extracranial: spinal canal",                                  "There is a masslike lesion centered along the right lateral spinal canal at the C1-2 level.", 'extracranial'),
    ("Extracranial: sinus content mid-paragraph",                   "Minimal scattered mucosal thickening involving the visualized paranasal sinuses. No orbital abnormality. The bones and extracranial soft tissues are otherwise unremarkable.", 'extracranial'),
    ("Extracranial: external carotid arteries",                     "External Carotid Arteries: Normal. No stenosis, occlusion or dissection.", 'extracranial'),
    ("Noise: study quality limited",                                "Study quality is limited secondary to motion artifact.", 'noise'),
    ("Noise: motion artifact",                                      "There is motion artifact, somewhat limiting evaluation.", 'noise'),
    ("Noise: ASL nondiagnostic",                                    "The ASL sequence is nondiagnostic and degraded by artifact.", 'noise'),
    ("Noise: removed from magnet",                                  "The patient was removed from the magnet prematurely before the completion of the majority of the examination.", 'noise'),
    ("Noise: website URL",                                          "You can find out more about our efforts to improve the [REDACTED] of radiology reports by visiting [REDACTED].", 'noise'),
    ("Extracranial: MRA finding",                                   "MRA [REDACTED]: No hemodynamically-significant stenosis of the innominate or bilateral common carotid arteries.", 'extracranial'),
    ("Extracranial: cauda equina / spine",                          "The conus terminates at the level of L1, in normal anatomic position. The cauda equina are normal.", 'extracranial'),
    ("Extracranial: vertebral artery dissection",                   "Left vertebral artery demonstrates multiple segments of narrowing and dilatation, highly suspicious for a vertebral artery dissection.", 'extracranial'),
    ("Extracranial: sagittal sinus laceration",                     "Loss of signal in the anterior portion of the superior sagittal sinus secondary to known laceration injury.", 'extracranial'),
    ("Negative: no regions of abnormal restricted diffusivity",     "There are no regions of abnormal restricted diffusivity demonstrated on the DWI.", 'negative'),
    ("Strip check: Brain Parenchyma: header",                       "Brain Parenchyma: Interval decrease in extent of T2/FLAIR abnormality surrounding the left basal ganglia resection cavity.", 'strip_check'),
    ("Negative: flow voids maintained",                             "The flow voids of the major intracranial vessels are maintained.", 'negative'),
    ("Noise: imaging quality degraded",                             "The imaging quality is degraded by patient motion.", 'noise'),
    ("Noise: suboptimal evaluation motion",                         "Suboptimal evaluation for infarction secondary to motion artifact and lack of diffusion-weighted images.", 'noise'),
    ("Extracranial: aortic arch / vertebral arteries arise",        "There is a conventional three-vessel aortic arch. The vertebral arteries arise from the subclavian arteries bilaterally.", 'extracranial'),
    ("Extracranial: parietal bone lesion",                          "The T2 hyperintense focus in the right parietal bone is unchanged.", 'extracranial'),
    ("Negative: otherwise bones unremarkable",                      "Otherwise, the bones and extracranial soft tissues are unremarkable.", 'negative'),
]

all_passed = True
for desc, text, expected in cases:
    if expected == 'strip_check':
        result = not strip_inline_header(text).startswith('Brain Parenchyma')
    elif expected == 'noise':          result = is_noise(text)
    elif expected == 'negative':       result = is_negative(text)
    elif expected == 'extracranial':   result = is_extracranial(text)
    status = 'PASS' if result else 'FAIL'
    if not result: all_passed = False
    print(f'  [{status}] {desc}')
print(f'\n{"All regression tests passed!" if all_passed else "Some tests FAILED -- fix patterns before updating repo."}')

# ── TEST 5 ────────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 5: 40 random positive brain findings (randomized each run)')
print('=' * 60)
pos = filtered[~filtered['is_negative'] & ~filtered['is_extracranial']]
for i, (_, row) in enumerate(pos.sample(40).iterrows()):
    print(f'\n[{i+1}] {row["paragraph"][:200]}')

print('\n' + '=' * 60)
print('ALL TESTS COMPLETE')
print('=' * 60)

Loading data...
Structural MRI rows: 36,924

TEST 1: Paragraph count — old vs new splitter
Old: mean=9.05 | median=8 | max=88 | >20para: 1,475 (4.0%)
New: mean=7.87 | median=7 | max=70 | >20para: 481 (1.3%)
Improved: 11,848 (32.1%) | Regressed: 0 (0.0%)

TEST 2: Multi-section report handling
Multi-section reports: 1,725 | Old avg: 19.6 | New avg: 6.9

TEST 3: Filter performance
Total paragraphs:            290,717
Dropped (noise):             8,838 (3.0%)
Remaining:                   281,879
  - Negative/normal:         69,150
  - Extracranial:            38,803
  - Positive brain findings: 187,992

TEST 4: Regression check (fixed cases)
  [PASS] Extracranial: 'The bones and extracranial...'
  [PASS] Noise: physician sign-off with [REDACTED]
  [PASS] Extracranial: partial opacification of sinuses
  [PASS] Negative: ventricles stable no hydrocephalus
  [PASS] Noise: post contrast technique note
  [PASS] Noise: per Epic note
  [PASS] Extracranial: dural sinuses
  [PASS] Negative: vascula

In [ ]:
import pandas as pd
import re

MRI_CSV = '../data/mri_reports.csv'

STRUCTURAL_TYPES = [
    'MRI BRAIN',
    'MRI BRAIN WITH AND WITHOUT CONTRAST',
    'MRI BRAIN WITHOUT CONTRAST',
]

# ── Filters v7 ────────────────────────────────────────────────────────────────

BOILERPLATE_PATTERN = re.compile(
    r'forwarded to an automated communication'
    r'|findings were (?:discussed|communicated)'
    r'|(?:the )?(?:critical )?findings in this report were reported to'
    r'|who responded indicating that the communication was understood'
    r'|alert notification of critical'
    r'|critical results were communicated'
    r'|this report has been'
    r'|electronically notify'
    r'|per epic note'
    r'|carotid stenosis reference'
    r'|(?:\[redacted\][\w\s,\.]*)?(?:md|do|phd|np|pa)[\s,]+the (?:attending|teaching) physician'
    r'|i,? the (?:attending|teaching) physician'
    r'|have reviewed the images and agree'
    r'|study (?:terminated|aborted|discontinued) (?:early|due to|at patient)'
    r'|post contrast[:\s]+following (?:iv|intravenous) administration'
    r'|following (?:iv|intravenous) administration of (?:gadolinium|contrast)'
    r'|(?:exam|study|images?|sequences?|imaging quality|image quality) (?:is|are) (?:limited|degraded) (?:by|due to|secondary to) (?:motion|artifact|patient)'
    r'|(?:patient was not able|patient could not) (?:to )?tolerate'
    r'|(?:patient was )?removed from the magnet'
    r'|only (?:diffusion|flair|dwi|t1|t2)[\w\s]+ (?:sequence|sequences|images?) (?:were|was) obtained'
    r'|(?:images?|sequences?) (?:are|were|is) degraded by'
    r'|pending postprocessing at the time of this dictation'
    r'|perfusion images? (?:are )?pending'
    r'|study quality is limited'
    r'|there is motion artifact'
    r'|(?:asl|dwi) sequence is nondiagnostic'
    r'|you can find out more about our efforts'
    r'|visiting \[redacted\]'
    r'|suboptimal evaluation (?:for|of|secondary to).{0,60}(?:motion|artifact|technique)'
    r'|follow.?up (?:is )?recommended'
    r'|contact was made at the time of interpr',
    re.IGNORECASE
)

HEADER_PATTERN = re.compile(
    r'^(?:brain mri|head mra|neck mra|mri brain|intracranial mra'
    r'|anterior circulation|posterior circulation|collateral circulation'
    r'|aortic\s+\[redacted\].*origin of major cervical'
    r'|impression|technique|comparison|indication|methodology'
    r'|detail|findings|neck'
    r'|ventricular system[\w\s\-]+:'
    r'|extra-axial spaces?:'
    r'|brain parenchyma:'
    r'|vascular system:'
    r'|vascular structures?:'
    r'|extracranial structures?:?'
    r'|flow.related signal'
    r'|synopsis for clinical management'
    r'|miscellaneous:'
    r'|calvarium:'
    r'|anterior circulation:):?\s*$',
    re.IGNORECASE
)

NEGATIVE_PATTERN = re.compile(
    r'^(?:there (?:is|are) )?(?:no evidence of|no acute|no new|no abnormal'
    r'|no significant|unremarkable|within normal limits|no hemorrhage'
    r'|no infarct|no mass effect|no hydrocephalus|no midline shift'
    r'|no space.occupying|no enhancing lesion|no abnormal enhancement'
    r'|no restricted diffusion|no acute intracranial|no brain parenchymal mass'
    r'|no interval change|no intracranial mass|no mass lesion)'
    r'|^(?:the )?(?:intracranial )?(?:major )?(?:flow.?voids?|arterial flow voids|vessels).{0,60}(?:intact|preserved|patent|maintained|present)'
    r'|^intracranial (?:major )?(?:dural sinus and )?arterial flow voids are present'
    r'|^flow.related signal is observed.{0,80}without (?:occlusion|stenosis)'
    r'|^the (?:visualized )?(?:paranasal sinuses|mastoid air cells?|orbits|bones and extracranial|extracranial soft tissues).{0,80}(?:clear|normal|unremarkable|unchanged)'
    r'|^the (?:remaining )?ventricles? (?:are|is) (?:stable|normal|unremarkable|unchanged)'
    r'|^(?:the )?ventricles?[,\s]+sulci[,\s]+and (?:cisterns?|basal cisterns?).{0,60}(?:unremarkable|normal|stable|within normal limits)'
    r'|^ventricles?[,\s]+sulci[,\s]+and cisterns? are stable'
    r'|^there has been no (?:significant )?interval change in (?:size|the size)'
    r'|^(?:\[redacted\]\s+)?venous sinuses? (?:enhance|are|appear).{0,40}(?:normal|patent|intact|homogenous)'
    r'|^vascular system:.{0,120}(?:intact|preserved|patent|normal|homogenous)'
    r'|^(?:extracranial structures|paranasal sinuses|mastoid).{0,80}(?:clear|normal|unremarkable)'
    r'|^(?:otherwise,?\s+)?the bones(?: and extracranial soft tissues)?.{0,60}(?:unremarkable|normal|unchanged)'
    r'|^(?:post contrast|following contrast).{0,60}no (?:abnormal|significant)'
    r'|^(?:there are )?no regions of abnormal restricted diffusivity'
    r'|^diffusion.weighted imaging demonstrates no acute'
    r'|^diffusion shows no signal abnormality'
    r'|^asl (?:sequence )?does not show'
    r'|^calvarium:.{0,60}(?:normal|within normal limits|unremarkable)',
    re.IGNORECASE
)

EXTRACRANIAL_PATTERN = re.compile(
    r'^(?:paranasal sinuses|mastoid air cells?|orbits|skull base'
    r'|extracranial soft tissues|cervical|vertebral arter'
    r'|aortic|subclavian|carotid|nasopharynx|external carotid'
    r'|mra\s+\[?redacted\]?)'
    r'|^(?:otherwise,?\s+)?the bones'
    r'|^the (?:visualized )?(?:paranasal sinuses|mastoid|orbits)'
    r'|^the (?:intracranial )?vertebral arteries'
    r'|^(?:partial|bilateral|mild|moderate|severe|minimal|scattered)?\s*(?:opacification|thickening|mucosal).{0,40}(?:sinus|mastoid|orbit)'
    r'|(?:mastoid air cells?|middle ear cav|maxillary sinus|sphenoid sinus|ethmoid sinus|paranasal sinus).{0,50}(?:opacif|thicken|fluid|effusion|clear|normal|air cell)'
    r'|(?:superior sagittal|transverse sinus|sigmoid sinus|straight sinus|cavernous sinus|dural (?:venous )?sinus|venous sinus).{0,80}(?:patent|thrombus|laceration|signal|enhance|filling defect|persistent)'
    r'|(?:nasopharynx|pharynx).{0,60}(?:edema|mucosal|thickening|fluid)'
    r'|(?:edema|mucosal thickening).{0,40}(?:nasopharynx|pharynx)'
    r'|(?:spinal canal|neuroforaminal|neural foramen|spinal cord|cauda equina|conus terminates|c\d-\d|l\d[ -]|cervical spine|thoracic spine|cervicomedullary)'
    r'|^(?:the )?(?:intracranial vertebral arteries|basilar artery).{0,40}(?:patent|normal)'
    r'|^(?:left|right)? vertebral artery (?:demonstrates|shows)'
    r'|^(?:the )?intracranial (?:ica|internal carotid).{0,80}(?:normal|patent|caliber)'
    r'|^(?:the )?(?:anterior|middle|posterior) cerebral arter.{0,60}(?:normal|patent|caliber)'
    r'|^(?:in )?the circle of.{0,80}(?:patent|atherosclerotic|anterior|posterior)'
    r'|^anterior circulation:'
    r'|^(?:no )?hemodynamically.significant stenosis'
    r'|(?:aortic arch|three.vessel arch|subclavian arter|vertebral arter(?:ies)? arise)'
    r'|(?:extracranial and intracranial|extracranial).{0,40}(?:internal carotid|carotid artery|vertebral)'
    r'|(?:flow.related enhancement|flow related enhancement).{0,60}(?:internal carotid|carotid|vertebral|middle cerebral)'
    r'|(?:\bright\b|\bleft\b|\bbilateral\b)?\s*(?:parietal bone|frontal bone|occipital bone|temporal bone|sphenoid bone|ethmoid bone)'
    r'|(?:calvarium|calvarial).{0,60}(?:focus|lesion|metastas|hyperintens|signal)'
    r'|(?:maxillary sinus|sphenoid sinus|ethmoid).{0,80}(?:asymmetr|diminutive|bowing|deviat|retention cyst)',
    re.IGNORECASE
)

INLINE_SUBSECTION_PATTERN = re.compile(
    r'^(?:brain parenchyma|ventricular system[\w\s\-]*|extra-axial spaces?'
    r'|vascular system|extracranial structures?|calvarium|miscellaneous'
    r'|synopsis for clinical management)\s*:\s*',
    re.IGNORECASE
)

def strip_inline_header(text):
    return INLINE_SUBSECTION_PATTERN.sub('', text).strip()

def is_noise(t):
    if len(t.strip()) < 30: return True
    if HEADER_PATTERN.match(t): return True
    if BOILERPLATE_PATTERN.search(t): return True
    return False

def is_negative(t): return bool(NEGATIVE_PATTERN.match(t))
def is_extracranial(t): return bool(EXTRACRANIAL_PATTERN.search(t))

# ── Splitter ──────────────────────────────────────────────────────────────────

SECTION_SPLIT_PATTERN = re.compile(
    r'\n(?=(?:BRAIN MRI|HEAD MRA|NECK MRA|MRI BRAIN AND NECK'
    r'|SPINE|CERVICAL|THORACIC|LUMBAR|STRUCTURAL MRI|FUNCTIONAL MRI)\s*[:\n])',
    re.IGNORECASE
)
BRAIN_SECTION_PATTERN = re.compile(r'^(?:BRAIN MRI|MRI BRAIN|STRUCTURAL MRI)', re.IGNORECASE)
INLINE_HEADER_PATTERN = re.compile(r'^(?:BRAIN MRI|HEAD MRA|MRI BRAIN)\s*:?\s*\n', re.IGNORECASE)

def _extract_brain_section(text):
    if not SECTION_SPLIT_PATTERN.search(text):
        return text
    sections = SECTION_SPLIT_PATTERN.split(text)
    brain = [s for s in sections if BRAIN_SECTION_PATTERN.match(s.strip())]
    return brain[0] if brain else sections[0]

def split_report(text):
    if pd.isna(text): return []
    text = _extract_brain_section(text)
    result = []
    for p in re.split(r'\n[ \t]*\n', text):
        p = INLINE_HEADER_PATTERN.sub('', p.strip()).strip()
        p = strip_inline_header(p)
        if len(p) >= 30:
            result.append(p)
    return result

def split_report_old(text):
    if pd.isna(text): return []
    return [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]

# ── Load ──────────────────────────────────────────────────────────────────────
print('Loading data...')
mri = pd.read_csv(MRI_CSV, engine='python', on_bad_lines='skip')
valid = mri[mri['findings'].notna() & ~mri['findings'].str.strip().str.lower().isin(['not available','none',''])]
structural = valid[valid['Type'].isin(STRUCTURAL_TYPES)].copy()
print(f'Structural MRI rows: {len(structural):,}\n')

# ── TEST 1 ────────────────────────────────────────────────────────────────────
print('=' * 60)
print('TEST 1: Paragraph count — old vs new splitter')
print('=' * 60)
structural['old_count'] = structural['findings'].apply(lambda t: len(split_report_old(t)))
structural['new_count'] = structural['findings'].apply(lambda t: len(split_report(t)))
for label, col in [('Old', 'old_count'), ('New', 'new_count')]:
    s = structural[col]
    print(f'{label}: mean={s.mean():.2f} | median={s.median():.0f} | max={s.max()} | >20para: {(s>20).sum():,} ({100*(s>20).mean():.1f}%)')
improved = (structural['new_count'] < structural['old_count']).sum()
regressed = (structural['new_count'] > structural['old_count']).sum()
print(f'Improved: {improved:,} ({100*improved/len(structural):.1f}%) | Regressed: {regressed:,} ({100*regressed/len(structural):.1f}%)')

# ── TEST 2 ────────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 2: Multi-section report handling')
print('=' * 60)
multi = structural[structural['findings'].str.contains(r'HEAD MRA|NECK MRA|MRI BRAIN AND NECK', case=False, regex=True, na=False)]
print(f'Multi-section reports: {len(multi):,} | Old avg: {multi["old_count"].mean():.1f} | New avg: {multi["new_count"].mean():.1f}')

# ── TEST 3 ────────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 3: Filter performance')
print('=' * 60)
structural['paragraph_list'] = structural['findings'].apply(split_report)
exploded = (
    structural.explode('paragraph_list')
    .rename(columns={'paragraph_list': 'paragraph'})
    .dropna(subset=['paragraph'])
    .reset_index(drop=True)
)
noise_mask = exploded['paragraph'].apply(is_noise)
filtered = exploded[~noise_mask].copy().reset_index(drop=True)
filtered['is_negative'] = filtered['paragraph'].apply(is_negative)
filtered['is_extracranial'] = filtered['paragraph'].apply(is_extracranial)
positive = (~filtered['is_negative'] & ~filtered['is_extracranial']).sum()
print(f'Total paragraphs:            {len(exploded):,}')
print(f'Dropped (noise):             {noise_mask.sum():,} ({100*noise_mask.mean():.1f}%)')
print(f'Remaining:                   {len(filtered):,}')
print(f'  - Negative/normal:         {filtered["is_negative"].sum():,}')
print(f'  - Extracranial:            {filtered["is_extracranial"].sum():,}')
print(f'  - Positive brain findings: {positive:,}')

# ── TEST 4 ────────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 4: Regression check (fixed cases)')
print('=' * 60)
cases = [
    ("Extracranial: 'The bones and extracranial...'",               "The bones and extracranial soft tissues are unchanged.", 'extracranial'),
    ("Noise: physician sign-off with [REDACTED]",                   "[REDACTED] MD, the teaching physician, have reviewed the images and agree with the report as written.", 'noise'),
    ("Extracranial: partial opacification of sinuses",              "Partial opacification of the maxillary sinuses bilaterally. The mastoid air cells and orbits are unremarkable.", 'extracranial'),
    ("Negative: ventricles stable no hydrocephalus",                "The remaining ventricles are stable in size and configuration. There is no hydrocephalus.", 'negative'),
    ("Noise: post contrast technique note",                         "Post contrast: Following IV administration of gadolinium chelate, no abnormal regions of enhancement are demonstrated.", 'noise'),
    ("Noise: per Epic note",                                        "Per Epic note, the patient subsequently underwent evacuation of the bifrontal fluid collection.", 'noise'),
    ("Extracranial: dural sinuses",                                 "The superior sagittal sinus, internal cerebral veins, inferior sagittal sinus, straight sinus are patent.", 'extracranial'),
    ("Negative: vascular system normal",                            "Vascular system: Intracranial major dural sinus and arterial flow voids are present. Venous sinuses are homogenous post contrast.", 'negative'),
    ("Extracranial: mastoid air cells mid-sentence",                "Patient status post left temporal craniotomy. Again opacification of left middle ear cavity and mastoid air cells.", 'extracranial'),
    ("Negative: ventricles sulci cisterns unremarkable",            "The ventricles, sulci, and basal cisterns are otherwise within normal limits.", 'negative'),
    ("Negative: ventricles sulci cisterns stable",                  "Ventricles, sulci, and cisterns are stable in size and configuration.", 'negative'),
    ("Negative: venous sinuses enhance normally",                   "[REDACTED] venous sinuses enhance normally.", 'negative'),
    ("Extracranial: nasopharynx mucosal thickening",                "There is mild edema and mucosal thickening in the nasopharynx anteriorly.", 'extracranial'),
    ("Extracranial: maxillary sinus asymmetry",                     "Again seen is an asymmetrically diminutive appearance of the right maxillary sinus with lateral bowing.", 'extracranial'),
    ("Noise: exam limited by motion artifact",                      "The axial images are limited due to patient motion artifact. Within this limitation, no high-grade spinal canal stenosis.", 'noise'),
    ("Noise: patient could not tolerate",                           "Only diffusion and FLAIR sequences were obtained as the patient was not able to tolerate further imaging.", 'noise'),
    ("Noise: perfusion pending",                                    "Of note, perfusion images are pending postprocessing at the time of this dictation.", 'noise'),
    ("Extracranial: spinal canal",                                  "There is a masslike lesion centered along the right lateral spinal canal at the C1-2 level.", 'extracranial'),
    ("Extracranial: sinus content mid-paragraph",                   "Minimal scattered mucosal thickening involving the visualized paranasal sinuses. No orbital abnormality. The bones and extracranial soft tissues are otherwise unremarkable.", 'extracranial'),
    ("Extracranial: external carotid arteries",                     "External Carotid Arteries: Normal. No stenosis, occlusion or dissection.", 'extracranial'),
    ("Noise: study quality limited",                                "Study quality is limited secondary to motion artifact.", 'noise'),
    ("Noise: motion artifact",                                      "There is motion artifact, somewhat limiting evaluation.", 'noise'),
    ("Noise: ASL nondiagnostic",                                    "The ASL sequence is nondiagnostic and degraded by artifact.", 'noise'),
    ("Noise: removed from magnet",                                  "The patient was removed from the magnet prematurely before the completion of the majority of the examination.", 'noise'),
    ("Noise: website URL",                                          "You can find out more about our efforts to improve the [REDACTED] of radiology reports by visiting [REDACTED].", 'noise'),
    ("Extracranial: MRA finding",                                   "MRA [REDACTED]: No hemodynamically-significant stenosis of the innominate or bilateral common carotid arteries.", 'extracranial'),
    ("Extracranial: cauda equina / spine",                          "The conus terminates at the level of L1, in normal anatomic position. The cauda equina are normal.", 'extracranial'),
    ("Extracranial: vertebral artery dissection",                   "Left vertebral artery demonstrates multiple segments of narrowing and dilatation, highly suspicious for a vertebral artery dissection.", 'extracranial'),
    ("Extracranial: sagittal sinus laceration",                     "Loss of signal in the anterior portion of the superior sagittal sinus secondary to known laceration injury.", 'extracranial'),
    ("Negative: no regions of abnormal restricted diffusivity",     "There are no regions of abnormal restricted diffusivity demonstrated on the DWI.", 'negative'),
    ("Strip check: Brain Parenchyma: header",                       "Brain Parenchyma: Interval decrease in extent of T2/FLAIR abnormality surrounding the left basal ganglia resection cavity.", 'strip_check'),
    ("Negative: flow voids maintained",                             "The flow voids of the major intracranial vessels are maintained.", 'negative'),
    ("Noise: imaging quality degraded",                             "The imaging quality is degraded by patient motion.", 'noise'),
    ("Noise: suboptimal evaluation motion",                         "Suboptimal evaluation for infarction secondary to motion artifact and lack of diffusion-weighted images.", 'noise'),
    ("Extracranial: aortic arch / vertebral arteries arise",        "There is a conventional three-vessel aortic arch. The vertebral arteries arise from the subclavian arteries bilaterally.", 'extracranial'),
    ("Extracranial: parietal bone lesion",                          "The T2 hyperintense focus in the right parietal bone is unchanged.", 'extracranial'),
    ("Negative: otherwise bones unremarkable",                      "Otherwise, the bones and extracranial soft tissues are unremarkable.", 'negative'),
    ("Noise: findings reported to clinician",                       "The findings in this report were reported to the referring clinician, Dr. [REDACTED] who responded indicating that the communication was understood.", 'noise'),
    ("Negative: intracranial flow voids present",                   "Intracranial major dural sinus and arterial flow voids are present.", 'negative'),
    ("Negative: flow related signal without occlusion",             "Flow-related signal is observed in the major intracranial arteries including the ICAs, vertebrobasilar arteries, bilateral MCA, without occlusion, or stenosis.", 'negative'),
    ("Extracranial: intracranial ICA normal caliber",               "The intracranial ICAs are normal in caliber and contour. The anterior and middle cerebral arteries are normal in caliber and contour.", 'extracranial'),
    ("Extracranial: circle of Willis",                              "In the circle of [REDACTED], distal internal carotid arteries are patent with mild atherosclerotic disease.", 'extracranial'),
    ("Extracranial: transverse sinus thrombus",                     "There has been interval resolution of previously identified thrombus within the left transverse sinus extending to the confluence of the sinus.", 'extracranial'),
    ("Extracranial: extracranial ICA flow attenuation",             "There is severe attenuation of flow related enhancement in the extracranial and intracranial right internal carotid artery.", 'extracranial'),
    ("Noise: follow up recommended",                                "Follow up recommended to insure stability.", 'noise'),
    ("Negative: diffusion shows no signal abnormality",             "Diffusion shows no signal abnormality in the brain.", 'negative'),
]

all_passed = True
for desc, text, expected in cases:
    if expected == 'strip_check':
        result = not strip_inline_header(text).startswith('Brain Parenchyma')
    elif expected == 'noise':          result = is_noise(text)
    elif expected == 'negative':       result = is_negative(text)
    elif expected == 'extracranial':   result = is_extracranial(text)
    status = 'PASS' if result else 'FAIL'
    if not result: all_passed = False
    print(f'  [{status}] {desc}')
print(f'\n{"All regression tests passed!" if all_passed else "Some tests FAILED -- fix patterns before updating repo."}')

# ── TEST 5 ────────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 5: 40 random positive brain findings (randomized each run)')
print('=' * 60)
pos = filtered[~filtered['is_negative'] & ~filtered['is_extracranial']]
for i, (_, row) in enumerate(pos.sample(40).iterrows()):
    print(f'\n[{i+1}] {row["paragraph"][:200]}')

print('\n' + '=' * 60)
print('ALL TESTS COMPLETE')
print('=' * 60)

In [12]:
# ── TARGETED FAILURE CHECK ────────────────────────────────────────────────────
print('=' * 60)
print('TARGETED FAILURE CHECK')
print('=' * 60)

failing_cases = [
    # Test 4 failures
    ("Negative: flow related signal without occlusion",
     "Flow-related signal is observed in the major intracranial arteries including the ICAs, vertebrobasilar arteries, bilateral MCA, without occlusion, or stenosis.",
     'negative'),
    ("Extracranial: transverse sinus thrombus",
     "There has been interval resolution of previously identified thrombus within the left transverse sinus extending to the confluence of the sinus.",
     'extracranial'),

    # Test 5 noise cases that slipped through
    ("Noise: closed loop communication system",
     "A clinically significant result was communicated and documented via a closed loop communication system.",
     'noise'),
    ("Noise: ANCR communication",
     "These results were communicated to the referring providers via ANCR.",
     'noise'),
    ("Noise: SWAN motion degraded",
     "SWAN and T2 star images are markedly motion degraded.",
     'noise'),
    ("Extracranial: nasoenteric tube oropharynx",
     "Nasoenteric tube appears coiled in the oropharynx after which appears to continue inferiorly into the pharynx.",
     'extracranial'),
    ("Negative: no gross abnormalities no diffusion abnormality",
     "No gross abnormalities No diffusion abnormality is seen.",
     'negative'),
]

all_passed = True
for desc, text, expected in failing_cases:
    if expected == 'noise':        result = is_noise(text)
    elif expected == 'negative':   result = is_negative(text)
    elif expected == 'extracranial': result = is_extracranial(text)
    status = 'PASS' if result else 'FAIL'
    if not result: all_passed = False
    print(f'  [{status}] {desc}')

print(f'\n{"All targeted tests passed!" if all_passed else "Some tests FAILED."}')

TARGETED FAILURE CHECK
  [FAIL] Negative: flow related signal without occlusion
  [FAIL] Extracranial: transverse sinus thrombus
  [FAIL] Noise: closed loop communication system
  [FAIL] Noise: ANCR communication
  [FAIL] Noise: SWAN motion degraded
  [FAIL] Extracranial: nasoenteric tube oropharynx
  [FAIL] Negative: no gross abnormalities no diffusion abnormality

Some tests FAILED.


In [13]:
import pandas as pd
import re

MRI_CSV = '../data/mri_reports.csv'

STRUCTURAL_TYPES = [
    'MRI BRAIN',
    'MRI BRAIN WITH AND WITHOUT CONTRAST',
    'MRI BRAIN WITHOUT CONTRAST',
]

# ── Filters v8 ────────────────────────────────────────────────────────────────

BOILERPLATE_PATTERN = re.compile(
    r'forwarded to an automated communication'
    r'|findings were (?:discussed|communicated)'
    r'|(?:the )?(?:critical )?findings in this report were reported to'
    r'|who responded indicating that the communication was understood'
    r'|alert notification of critical'
    r'|critical results were communicated'
    r'|this report has been'
    r'|electronically notify'
    r'|per epic note'
    r'|carotid stenosis reference'
    r'|(?:\[redacted\][\w\s,\.]*)?(?:md|do|phd|np|pa)[\s,]+the (?:attending|teaching) physician'
    r'|i,? the (?:attending|teaching) physician'
    r'|have reviewed the images and agree'
    r'|study (?:terminated|aborted|discontinued) (?:early|due to|at patient)'
    r'|post contrast[:\s]+following (?:iv|intravenous) administration'
    r'|following (?:iv|intravenous) administration of (?:gadolinium|contrast)'
    r'|(?:exam|study|images?|sequences?|imaging quality|image quality) (?:is|are) (?:limited|degraded) (?:by|due to|secondary to) (?:motion|artifact|patient)'
    r'|(?:patient was not able|patient could not) (?:to )?tolerate'
    r'|(?:patient was )?removed from the magnet'
    r'|only (?:diffusion|flair|dwi|t1|t2)[\w\s]+ (?:sequence|sequences|images?) (?:were|was) obtained'
    r'|(?:images?|sequences?) (?:are|were|is) degraded by'
    r'|pending postprocessing at the time of this dictation'
    r'|perfusion images? (?:are )?pending'
    r'|study quality is limited'
    r'|there is motion artifact'
    r'|(?:asl|dwi) sequence is nondiagnostic'
    r'|you can find out more about our efforts'
    r'|visiting \[redacted\]'
    r'|suboptimal evaluation (?:for|of|secondary to).{0,60}(?:motion|artifact|technique)'
    r'|follow.?up (?:is )?recommended'
    r'|contact was made at the time of interpr'
    r'|(?:communicated|documented) via(?: a)? closed.?loop communication'
    r'|(?:results? were|findings? were) communicated to the referring (?:provider|physician|clinician)s? via'
    r'|communicated to the referring providers? via \w+'
    r'|(?:swan|t2 star|t2\*).{0,40}(?:motion degraded|markedly.{0,10}degraded|nondiagnostic)',
    re.IGNORECASE
)

HEADER_PATTERN = re.compile(
    r'^(?:brain mri|head mra|neck mra|mri brain|intracranial mra'
    r'|anterior circulation|posterior circulation|collateral circulation'
    r'|aortic\s+\[redacted\].*origin of major cervical'
    r'|impression|technique|comparison|indication|methodology'
    r'|detail|findings|neck'
    r'|ventricular system[\w\s\-]+:'
    r'|extra-axial spaces?:'
    r'|brain parenchyma:'
    r'|vascular system:'
    r'|vascular structures?:'
    r'|extracranial structures?:?'
    r'|flow.related signal'
    r'|synopsis for clinical management'
    r'|miscellaneous:'
    r'|calvarium:'
    r'|anterior circulation:):?\s*$',
    re.IGNORECASE
)

NEGATIVE_PATTERN = re.compile(
    r'^(?:there (?:is|are) )?(?:no evidence of|no acute|no new|no abnormal'
    r'|no significant|unremarkable|within normal limits|no hemorrhage'
    r'|no infarct|no mass effect|no hydrocephalus|no midline shift'
    r'|no space.occupying|no enhancing lesion|no abnormal enhancement'
    r'|no restricted diffusion|no acute intracranial|no brain parenchymal mass'
    r'|no interval change|no intracranial mass|no mass lesion'
    r'|no gross abnormalities)'
    r'|^(?:the )?(?:intracranial )?(?:major )?(?:flow.?voids?|arterial flow voids|vessels).{0,60}(?:intact|preserved|patent|maintained|present)'
    r'|^intracranial (?:major )?(?:dural sinus and )?arterial flow voids are present'
    r'|^flow.related signal is observed.{0,120}without (?:occlusion|stenosis)'
    r'|^the (?:visualized )?(?:paranasal sinuses|mastoid air cells?|orbits|bones and extracranial|extracranial soft tissues).{0,80}(?:clear|normal|unremarkable|unchanged)'
    r'|^the (?:remaining )?ventricles? (?:are|is) (?:stable|normal|unremarkable|unchanged)'
    r'|^(?:the )?ventricles?[,\s]+sulci[,\s]+and (?:cisterns?|basal cisterns?).{0,60}(?:unremarkable|normal|stable|within normal limits)'
    r'|^ventricles?[,\s]+sulci[,\s]+and cisterns? are stable'
    r'|^there has been no (?:significant )?interval change in (?:size|the size)'
    r'|^(?:\[redacted\]\s+)?venous sinuses? (?:enhance|are|appear).{0,40}(?:normal|patent|intact|homogenous)'
    r'|^vascular system:.{0,120}(?:intact|preserved|patent|normal|homogenous)'
    r'|^(?:extracranial structures|paranasal sinuses|mastoid).{0,80}(?:clear|normal|unremarkable)'
    r'|^(?:otherwise,?\s+)?the bones(?: and extracranial soft tissues)?.{0,60}(?:unremarkable|normal|unchanged)'
    r'|^(?:post contrast|following contrast).{0,60}no (?:abnormal|significant)'
    r'|^(?:there are )?no regions of abnormal restricted diffusivity'
    r'|^diffusion.weighted imaging demonstrates no acute'
    r'|^diffusion shows no signal abnormality'
    r'|^asl (?:sequence )?does not show'
    r'|^calvarium:.{0,60}(?:normal|within normal limits|unremarkable)',
    re.IGNORECASE
)

EXTRACRANIAL_PATTERN = re.compile(
    r'^(?:paranasal sinuses|mastoid air cells?|orbits|skull base'
    r'|extracranial soft tissues|cervical|vertebral arter'
    r'|aortic|subclavian|carotid|nasopharynx|external carotid'
    r'|mra\s+\[?redacted\]?)'
    r'|^(?:otherwise,?\s+)?the bones'
    r'|^the (?:visualized )?(?:paranasal sinuses|mastoid|orbits)'
    r'|^the (?:intracranial )?vertebral arteries'
    r'|^(?:partial|bilateral|mild|moderate|severe|minimal|scattered)?\s*(?:opacification|thickening|mucosal).{0,40}(?:sinus|mastoid|orbit)'
    r'|(?:mastoid air cells?|middle ear cav|maxillary sinus|sphenoid sinus|ethmoid sinus|paranasal sinus).{0,50}(?:opacif|thicken|fluid|effusion|clear|normal|air cell)'
    r'|(?:superior sagittal|transverse sinus|sigmoid sinus|straight sinus|cavernous sinus|dural (?:venous )?sinus|venous sinus).{0,80}(?:patent|thrombus|laceration|signal|enhance|filling defect|persistent|resolution|confluence)'
    r'|(?:nasopharynx|pharynx).{0,60}(?:edema|mucosal|thickening|fluid)'
    r'|(?:edema|mucosal thickening).{0,40}(?:nasopharynx|pharynx)'
    r'|(?:spinal canal|neuroforaminal|neural foramen|spinal cord|cauda equina|conus terminates|c\d-\d|l\d[ -]|cervical spine|thoracic spine|cervicomedullary)'
    r'|^(?:the )?(?:intracranial vertebral arteries|basilar artery).{0,40}(?:patent|normal)'
    r'|^(?:left|right)? vertebral artery (?:demonstrates|shows)'
    r'|^(?:the )?intracranial (?:ica|internal carotid).{0,80}(?:normal|patent|caliber)'
    r'|^(?:the )?(?:anterior|middle|posterior) cerebral arter.{0,60}(?:normal|patent|caliber)'
    r'|^(?:in )?the circle of.{0,80}(?:patent|atherosclerotic|anterior|posterior)'
    r'|^anterior circulation:'
    r'|^(?:no )?hemodynamically.significant stenosis'
    r'|(?:aortic arch|three.vessel arch|subclavian arter|vertebral arter(?:ies)? arise)'
    r'|(?:extracranial and intracranial|extracranial).{0,40}(?:internal carotid|carotid artery|vertebral)'
    r'|(?:flow.related enhancement|flow related enhancement).{0,60}(?:internal carotid|carotid|vertebral|middle cerebral)'
    r'|(?:\bright\b|\bleft\b|\bbilateral\b)?\s*(?:parietal bone|frontal bone|occipital bone|temporal bone|sphenoid bone|ethmoid bone)'
    r'|(?:calvarium|calvarial).{0,60}(?:focus|lesion|metastas|hyperintens|signal)'
    r'|(?:maxillary sinus|sphenoid sinus|ethmoid).{0,80}(?:asymmetr|diminutive|bowing|deviat|retention cyst)'
    r'|(?:nasogastric|nasoenteric|feeding).{0,30}(?:tube|ng tube).{0,60}(?:oropharynx|pharynx|stomach|esophagus|coiled|positioned)'
    r'|(?:oropharynx|pharynx).{0,60}(?:tube|coiled)',
    re.IGNORECASE
)

INLINE_SUBSECTION_PATTERN = re.compile(
    r'^(?:brain parenchyma|ventricular system[\w\s\-]*|extra-axial spaces?'
    r'|vascular system|extracranial structures?|calvarium|miscellaneous'
    r'|synopsis for clinical management)\s*:\s*',
    re.IGNORECASE
)

def strip_inline_header(text):
    return INLINE_SUBSECTION_PATTERN.sub('', text).strip()

def is_noise(t):
    if len(t.strip()) < 30: return True
    if HEADER_PATTERN.match(t): return True
    if BOILERPLATE_PATTERN.search(t): return True
    return False

def is_negative(t): return bool(NEGATIVE_PATTERN.match(t))
def is_extracranial(t): return bool(EXTRACRANIAL_PATTERN.search(t))

# ── Splitter ──────────────────────────────────────────────────────────────────

SECTION_SPLIT_PATTERN = re.compile(
    r'\n(?=(?:BRAIN MRI|HEAD MRA|NECK MRA|MRI BRAIN AND NECK'
    r'|SPINE|CERVICAL|THORACIC|LUMBAR|STRUCTURAL MRI|FUNCTIONAL MRI)\s*[:\n])',
    re.IGNORECASE
)
BRAIN_SECTION_PATTERN = re.compile(r'^(?:BRAIN MRI|MRI BRAIN|STRUCTURAL MRI)', re.IGNORECASE)
INLINE_HEADER_PATTERN = re.compile(r'^(?:BRAIN MRI|HEAD MRA|MRI BRAIN)\s*:?\s*\n', re.IGNORECASE)

def _extract_brain_section(text):
    if not SECTION_SPLIT_PATTERN.search(text):
        return text
    sections = SECTION_SPLIT_PATTERN.split(text)
    brain = [s for s in sections if BRAIN_SECTION_PATTERN.match(s.strip())]
    return brain[0] if brain else sections[0]

def split_report(text):
    if pd.isna(text): return []
    text = _extract_brain_section(text)
    result = []
    for p in re.split(r'\n[ \t]*\n', text):
        p = INLINE_HEADER_PATTERN.sub('', p.strip()).strip()
        p = strip_inline_header(p)
        if len(p) >= 30:
            result.append(p)
    return result

def split_report_old(text):
    if pd.isna(text): return []
    return [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]

# ── Load ──────────────────────────────────────────────────────────────────────
print('Loading data...')
mri = pd.read_csv(MRI_CSV, engine='python', on_bad_lines='skip')
valid = mri[mri['findings'].notna() & ~mri['findings'].str.strip().str.lower().isin(['not available','none',''])]
structural = valid[valid['Type'].isin(STRUCTURAL_TYPES)].copy()
print(f'Structural MRI rows: {len(structural):,}\n')

# ── TARGETED FAILURE CHECK ────────────────────────────────────────────────────
print('=' * 60)
print('TARGETED FAILURE CHECK')
print('=' * 60)

failing_cases = [
    ("Negative: flow related signal without occlusion",
     "Flow-related signal is observed in the major intracranial arteries including the ICAs, vertebrobasilar arteries, bilateral MCA, without occlusion, or stenosis.",
     'negative'),
    ("Extracranial: transverse sinus thrombus",
     "There has been interval resolution of previously identified thrombus within the left transverse sinus extending to the confluence of the sinus.",
     'extracranial'),
    ("Noise: closed loop communication system",
     "A clinically significant result was communicated and documented via a closed loop communication system.",
     'noise'),
    ("Noise: ANCR communication",
     "These results were communicated to the referring providers via ANCR.",
     'noise'),
    ("Noise: SWAN motion degraded",
     "SWAN and T2 star images are markedly motion degraded.",
     'noise'),
    ("Extracranial: nasoenteric tube oropharynx",
     "Nasoenteric tube appears coiled in the oropharynx after which appears to continue inferiorly into the pharynx.",
     'extracranial'),
    ("Negative: no gross abnormalities no diffusion abnormality",
     "No gross abnormalities No diffusion abnormality is seen.",
     'negative'),
]

all_passed = True
for desc, text, expected in failing_cases:
    if expected == 'noise':          result = is_noise(text)
    elif expected == 'negative':     result = is_negative(text)
    elif expected == 'extracranial': result = is_extracranial(text)
    status = 'PASS' if result else 'FAIL'
    if not result: all_passed = False
    print(f'  [{status}] {desc}')

print(f'\n{"All targeted tests passed!" if all_passed else "Some tests FAILED."}')

Loading data...
Structural MRI rows: 36,924

TARGETED FAILURE CHECK
  [PASS] Negative: flow related signal without occlusion
  [PASS] Extracranial: transverse sinus thrombus
  [PASS] Noise: closed loop communication system
  [PASS] Noise: ANCR communication
  [PASS] Noise: SWAN motion degraded
  [PASS] Extracranial: nasoenteric tube oropharynx
  [PASS] Negative: no gross abnormalities no diffusion abnormality

All targeted tests passed!
